# Python Bug Detection Using CodeBERT on the BugInPy Dataset

**PhD topic:** Large Language Models for Automated Program Debugging
**Experiment 1:** Binary classification of Python code units — `1 = buggy`, `0 = non-buggy` — by fine-tuning `microsoft/codebert-base` on BugInPy.

---

## How to use this notebook

* This notebook is designed for **Google Colab** (`Runtime → Change runtime type → GPU`, e.g. T4).
* Run it **cell by cell**. Do **not** use `Runtime → Run all`.
* Every section follows the same structure:
  1. **Explanation** — what the step does.
  2. **Scientific objective** — why the step matters for the validity of the experiment.
  3. **Code** — the cell(s) to execute.
  4. **Expected output** — what you should see.
  5. **Verification** — an automated check (`assert`) that fails loudly if something is wrong.
  6. **Interpretation** — how to read the result.
* No results in this notebook are pre-filled. All numbers come from your own execution.

## Experimental phases

| Phase | Title | Status |
|---|---|---|
| 1 | Environment setup | done |
| 2 | BugInPy download | done |
| 3 | Dataset exploration | done |
| 4 | Dataset construction (labeling strategy) | done |
| 5 | Data preprocessing & tokenization | done |
| 6 | Leakage-free dataset splitting (5-fold cross-project CV) | **available** |
| 7 | Baselines (TF-IDF + LR / RF, length-only shortcut check) | pending |
| 8 | CodeBERT fine-tuning | pending |
| 9 | Evaluation | pending |
| 10 | Scientific analysis & threats to validity | pending |

Phases are added to this notebook incrementally, after the results of the previous phase are confirmed.

---
# PHASE 1 — Environment Setup

**Goal of the phase:** build a known, documented and reproducible computational environment before touching any data.

Sections:
* 1.1 Python version and global reproducibility flags
* 1.2 GPU availability and hardware information
* 1.3 Installation of required libraries
* 1.4 Imports and library versions
* 1.5 Random seeds and determinism
* 1.6 (Optional) Google Drive mounting and experiment directory layout
* 1.7 Experiment configuration and environment record
* 1.8 Phase 1 verification summary

## 1.1 Python version and global reproducibility flags

**Explanation.** We print the Python interpreter version and platform, and set environment variables that must exist *before* PyTorch initialises CUDA.

**Scientific objective.** The software stack is part of the experimental setup. Recording it allows other researchers (and reviewers) to reproduce the run. `CUBLAS_WORKSPACE_CONFIG` is required by PyTorch for deterministic cuBLAS operations; `PYTHONHASHSEED` fixes hashing in this process (e.g. ordering of `set` iteration in child processes).

**Expected output.** A Python version ≥ 3.9 (Colab currently ships Python 3.10–3.12), the platform string, and confirmation that the variables are set.

In [ ]:
# ---- 1.1 Python version & reproducibility flags ----
import os, sys, platform

# Must be set BEFORE torch initialises CUDA (i.e. before the first `import torch`).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # avoids fork warnings from HF tokenizers

IN_COLAB = "google.colab" in sys.modules

print("Python version :", sys.version.split()[0])
print("Implementation :", platform.python_implementation())
print("Platform       :", platform.platform())
print("Running in Colab:", IN_COLAB)
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

# ---- Verification ----
assert sys.version_info >= (3, 9), "Python >= 3.9 is required."
if not IN_COLAB:
    print("WARNING: this notebook is designed for Google Colab; some cells (Drive) will be skipped.")
print("\n[OK] 1.1 passed")

**Interpretation.** If `Running in Colab` is `False`, you are not in Colab (e.g. local Jupyter); the notebook still works but Drive integration is skipped. Note the Python version — it will be stored automatically in the environment record (1.7).

## 1.2 GPU availability and hardware information

**Explanation.** We query the NVIDIA driver (`nvidia-smi`) and PyTorch's view of CUDA, and record CPU / RAM / disk.

**Scientific objective.** Training time, maximum batch size and even small numerical differences depend on the hardware. Fine-tuning CodeBERT (≈125 M parameters) on CPU is impractically slow, so a GPU is strongly recommended for Phase 8. Baselines (Phase 7) can run on CPU.

**Expected output.**
* With a GPU runtime: an `nvidia-smi` table (typically *Tesla T4, 15 GB*; on Colab Pro possibly L4 / A100), `CUDA available: True`, the GPU name and memory.
* Without a GPU: `CUDA available: False` and a warning. In that case go to **Runtime → Change runtime type → T4 GPU**, then restart from cell 1.1.

In [ ]:
# ---- 1.2 GPU & hardware information ----
import subprocess, shutil

def run(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f"<failed: {e}>"

print("=== nvidia-smi ===")
print(run("nvidia-smi") or "nvidia-smi not available (no GPU attached).")

import torch
cuda_ok = torch.cuda.is_available()
print("\n=== PyTorch view ===")
print("torch version   :", torch.__version__)
print("CUDA available  :", cuda_ok)
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print("CUDA (torch)    :", torch.version.cuda)
    print("cuDNN           :", torch.backends.cudnn.version())
    print("GPU count       :", torch.cuda.device_count())
    print("GPU name        :", props.name)
    print(f"GPU memory      : {props.total_memory / 1024**3:.1f} GB")
    print(f"Compute capab.  : {props.major}.{props.minor}")
    print("bf16 supported  :", torch.cuda.is_bf16_supported())

print("\n=== CPU / RAM / Disk ===")
print("CPU             :", run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'") or platform.processor())
print("Logical cores   :", os.cpu_count())
print("RAM             :", run("free -h | awk '/Mem:/ {print $2\" total, \"$7\" available\"}'"))
total, used, free = shutil.disk_usage("/")
print(f"Disk (/)        : {total/1024**3:.0f} GB total, {free/1024**3:.0f} GB free")

DEVICE = torch.device("cuda" if cuda_ok else "cpu")
print("\nSelected DEVICE :", DEVICE)

# ---- Verification ----
if not cuda_ok:
    print("\nWARNING: No GPU detected. Phases 1-7 can run on CPU, but Phase 8 (CodeBERT) "
          "will be extremely slow. Switch to a GPU runtime before Phase 8.")
else:
    x = torch.randn(1024, 1024, device=DEVICE)
    assert torch.isfinite(x @ x).all(), "GPU matmul sanity check failed."
    print("GPU matmul sanity check: OK")
print("\n[OK] 1.2 passed")

**Interpretation.** Note the GPU model and memory: with a 15 GB T4, CodeBERT with `max_length=512` fits comfortably with batch size 8–16 (mixed precision). The GPU name is stored in the environment record, and will be reported together with training time in Phase 8.

## 1.3 Installation of required libraries

**Explanation.** Colab already provides `torch`, `numpy`, `pandas`, `scikit-learn` and `matplotlib`. We install/upgrade the Hugging Face stack and a few utilities. We deliberately **do not reinstall PyTorch**, to keep the CUDA build that matches Colab's driver.

| Library | Role in the experiment |
|---|---|
| `transformers` | CodeBERT model + tokenizer, training utilities |
| `datasets` | efficient tokenized dataset storage |
| `accelerate` | required by `transformers` Trainer / mixed precision |
| `evaluate` | standard metric implementations (cross-check) |
| `scikit-learn` | TF-IDF baselines, metrics, splitting utilities |
| `seaborn`, `matplotlib` | figures |
| `tqdm` | progress bars |
| `GitPython` | inspecting BugInPy git metadata (Phase 2–4) |

**Scientific objective.** A controlled and recorded dependency set. Exact installed versions are logged in 1.4 and saved in 1.7.

**Expected output.** pip progress (quiet mode prints little). Possible *dependency-resolver warnings* about unrelated Colab packages are normally harmless. If pip reports that a **restart is required**, use `Runtime → Restart session`, then re-run from **1.1** (skip 1.3).

In [ ]:
# ---- 1.3 Install libraries (run once per Colab session) ----
%pip install -q -U "transformers>=4.44,<5" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" \
                   "scikit-learn>=1.3" seaborn tqdm GitPython
print("\n[OK] 1.3 installation finished — check above for errors (warnings are usually fine).")

**Verification.** The next cell (1.4) imports every library and prints its version; an `ImportError` there means an installation failed.

**Interpretation.** Version ranges (not exact pins) are used because Colab images change over time; the *exact* versions actually used are what matters for reproducibility, and those are recorded.

## 1.4 Imports and library versions

**Explanation.** Import every dependency used across all phases and print the versions.

**Scientific objective.** Fail early if anything is missing, and document the precise software versions.

**Expected output.** A table of library versions, e.g. `transformers 4.4x.x`, `torch 2.x.x+cu12x`, and `[OK] 1.4 passed`.

In [ ]:
# ---- 1.4 Imports ----
import json, time, random, re, ast, hashlib, textwrap, warnings, datetime, logging
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import sklearn
import transformers
import datasets
import accelerate
import evaluate
import git  # GitPython

from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)

LIB_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "evaluate": evaluate.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "GitPython": git.__version__,
}
print(pd.Series(LIB_VERSIONS, name="version").to_string())

# ---- Verification ----
assert int(transformers.__version__.split(".")[0]) == 4, "Expected transformers 4.x"
print("\n[OK] 1.4 passed")

**Interpretation.** These versions are saved to `environment.json` in 1.7 and must be reported in the methodology section of the paper/thesis.

## 1.5 Random seeds and determinism

**Explanation.** Fix the seeds of Python `random`, NumPy and PyTorch (CPU + CUDA), and enable deterministic algorithms.

**Scientific objective.** Fine-tuning results vary with weight initialisation of the classification head, data shuffling and dropout. Fixing seeds makes a single run repeatable. *Note:* full bit-level determinism on GPU is not always guaranteed (some CUDA kernels are non-deterministic); for the final thesis results we will therefore also report the **mean ± std over several seeds** (planned in Phase 8/9).

**Expected output.** `Seed set to 42`, followed by two identical random draws (re-seeding test) and `[OK] 1.5 passed`.

In [ ]:
# ---- 1.5 Seeds ----
SEED = 42

def set_seed(seed: int = SEED, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # warn_only=True: report (not crash on) ops without a deterministic implementation
        torch.use_deterministic_algorithms(True, warn_only=True)
    print(f"Seed set to {seed} (deterministic={deterministic})")

set_seed(SEED)

# ---- Verification: re-seeding must reproduce the same numbers ----
def draw():
    return (random.random(), float(np.random.rand()), float(torch.rand(1)))
set_seed(SEED); a = draw()
set_seed(SEED); b = draw()
print("draw 1:", a)
print("draw 2:", b)
assert a == b, "Seeding is not reproducible!"
set_seed(SEED)
print("\n[OK] 1.5 passed")

**Interpretation.** Identical draws confirm that every random generator used by the pipeline is controlled by `SEED`. The seed value is recorded in the configuration (1.7).

## 1.6 (Optional) Google Drive mounting and directory layout

**Explanation.** Colab's local disk is erased when the session ends. Mounting Google Drive lets us persist the processed dataset, models, checkpoints, logs, metrics and figures. Set `USE_DRIVE = False` to work only on the temporary local disk.

When the cell runs, Colab will open an authorisation pop-up — accept access to your Drive.

**Directory layout** (created under `MyDrive/PhD_Experiments/BugInPy_CodeBERT/`):

```
BugInPy_CodeBERT/
├── data/
│   ├── raw/          # untouched copy of BugInPy (Phase 2) — never modified
│   ├── interim/      # extracted code units before cleaning (Phase 4)
│   └── processed/    # final labelled dataset + split files (Phases 5-6)
├── models/
│   ├── baselines/    # TF-IDF + LR / RF (Phase 7)
│   └── codebert/     # final fine-tuned model + tokenizer (Phase 8)
├── checkpoints/      # intermediate training checkpoints
├── logs/             # training logs, environment & config records
├── results/
│   ├── metrics/      # CSV / JSON evaluation results
│   └── predictions/  # per-sample predictions (error analysis)
└── figures/          # plots (loss curves, ROC, confusion matrix ...)
```

**Scientific objective.** Persistent, organised storage of every artefact, which is required for reproducibility and for later phases of the PhD.

**Expected output.** `Mounted at /content/drive` (if `USE_DRIVE=True`) and a listing of the created folders.

In [ ]:
# ---- 1.6 Optional Google Drive + directory layout ----
USE_DRIVE = True          # <-- set to False to keep everything on the temporary Colab disk
EXPERIMENT_NAME = "BugInPy_CodeBERT"

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/PhD_Experiments") / EXPERIMENT_NAME
else:
    if USE_DRIVE and not IN_COLAB:
        print("Not running in Colab: Drive unavailable, using local directory.")
    BASE_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / EXPERIMENT_NAME

# Fast local working dir (git clone etc. is much faster on local disk than on Drive)
WORK_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / "work"

PATHS = {
    "base":        BASE_DIR,
    "raw":         BASE_DIR / "data" / "raw",
    "interim":     BASE_DIR / "data" / "interim",
    "processed":   BASE_DIR / "data" / "processed",
    "baselines":   BASE_DIR / "models" / "baselines",
    "codebert":    BASE_DIR / "models" / "codebert",
    "checkpoints": BASE_DIR / "checkpoints",
    "logs":        BASE_DIR / "logs",
    "metrics":     BASE_DIR / "results" / "metrics",
    "predictions": BASE_DIR / "results" / "predictions",
    "figures":     BASE_DIR / "figures",
    "work":        WORK_DIR,
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    print(f"{k:<12} -> {p}")

# ---- Verification: every directory exists and is writable ----
for k, p in PATHS.items():
    assert p.is_dir(), f"Missing dir: {p}"
    probe = p / ".write_test"
    probe.write_text("ok"); probe.unlink()
print("\nStorage persistent across sessions:", USE_DRIVE and IN_COLAB)
print("[OK] 1.6 passed")

**Interpretation.** If `Storage persistent across sessions: False`, all outputs will be lost when the Colab runtime is recycled — acceptable for testing, not for final experiments. `WORK_DIR` (local disk) is used for heavy temporary operations; final artefacts are always copied to `BASE_DIR`.

## 1.7 Experiment configuration and environment record

**Explanation.** A single `CONFIG` dictionary centralises every experimental choice (seed, model, hyperparameters, split ratios). Later phases read from it — nothing is hard-coded elsewhere. The configuration and full environment description are saved as JSON with a unique **run ID**.

**Scientific objective.** Research reproducibility: seed, dataset version, model name, hyperparameters and hardware are all recorded. Fields marked `None` are filled in by later phases (e.g. the exact BugInPy commit hash in Phase 2, split identifiers in Phase 6, training time in Phase 8).

**Expected output.** The printed configuration and the paths of `config_<RUN_ID>.json` and `environment_<RUN_ID>.json` in `logs/`.

In [ ]:
# ---- 1.7 Configuration & environment record ----
# After a Colab restart, put your previous RUN_ID here to continue the SAME run
# (e.g. RESUME_RUN_ID = "20261112_122126"). Leave None to start a new run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "run_id": RUN_ID,
    "experiment": EXPERIMENT_NAME,
    "seed": SEED,
    # --- dataset (filled in Phase 2) ---
    "dataset": {
        "name": "BugInPy",
        "repository": "https://github.com/soarsmu/BugsInPy",
        "commit": None,          # exact commit hash recorded in Phase 2
    },
    # --- model ---
    "model_name": "microsoft/codebert-base",
    "num_labels": 2,
    "label_map": {"0": "non-buggy", "1": "buggy"},
    # --- training hyperparameters (Phase 8) ---
    "hyperparameters": {
        "optimizer": "AdamW",
        "learning_rate": 2e-5,
        "batch_size": 16 if cuda_ok else 8,
        "num_epochs": 5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "max_seq_length": 512,
        "early_stopping_patience": 2,
        "early_stopping_metric": "eval_f1",
        "loss": "CrossEntropyLoss",
        "fp16": bool(cuda_ok),
    },
    # --- splitting (Phase 6) ---
    "split": {
        "strategy": "project-level (grouped)",
        "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
        "split_ids_file": None,  # filled in Phase 6
    },
    "paths": {k: str(v) for k, v in PATHS.items()},
}

ENVIRONMENT = {
    "run_id": RUN_ID,
    "timestamp": datetime.datetime.now().isoformat(),
    "in_colab": IN_COLAB,
    "platform": platform.platform(),
    "libraries": LIB_VERSIONS,
    "hardware": {
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if cuda_ok else None,
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda_ok else None,
        "cuda_version": torch.version.cuda if cuda_ok else None,
        "cudnn_version": torch.backends.cudnn.version() if cuda_ok else None,
        "cpu_count": os.cpu_count(),
        "cpu_model": run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'"),
    },
    "pip_freeze_file": f"pip_freeze_{RUN_ID}.txt",
}

def save_json(obj, path):
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, default=str))
    return path

cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
env_path = save_json(ENVIRONMENT, PATHS["logs"] / f"environment_{RUN_ID}.json")
(PATHS["logs"] / ENVIRONMENT["pip_freeze_file"]).write_text(run(f"{sys.executable} -m pip freeze"))

print(json.dumps(CONFIG, indent=2)[:2500])
print("\nSaved:", cfg_path)
print("Saved:", env_path)
print("Saved:", PATHS["logs"] / ENVIRONMENT["pip_freeze_file"])

# ---- Verification: files can be read back and match ----
assert json.loads(cfg_path.read_text())["seed"] == SEED
assert json.loads(env_path.read_text())["libraries"]["transformers"] == transformers.__version__
print("\n[OK] 1.7 passed")

**Interpretation.** `RUN_ID` uniquely identifies this experimental run; every file produced later will carry it. The `pip_freeze` file is the most precise record of the software environment and can be used to rebuild it exactly.

> **Note on hyperparameters.** The values above follow your specification (AdamW, lr = 2e-5, batch 8/16, 5 epochs, early stopping, cross-entropy). `weight_decay=0.01`, `warmup_ratio=0.1` and `max_seq_length=512` are standard defaults for CodeBERT fine-tuning; they will be discussed again in Phase 5 (sequence length) and Phase 8 before training.

## 1.8 Phase 1 verification summary

**Explanation.** A final checklist of the whole phase.

**Expected output.** A table where every row says `PASS` (the GPU row may say `WARN` on a CPU runtime).

In [ ]:
# ---- 1.8 Phase 1 summary ----
checks = {
    "Python >= 3.9":                 "PASS" if sys.version_info >= (3, 9) else "FAIL",
    "GPU available":                 "PASS" if cuda_ok else "WARN (CPU only)",
    "transformers importable":       "PASS",
    "Seeds reproducible":            "PASS",
    "Output directories writable":   "PASS",
    "Persistent storage (Drive)":    "PASS" if (USE_DRIVE and IN_COLAB) else "WARN (temporary disk)",
    "Config + environment saved":    "PASS" if cfg_path.exists() and env_path.exists() else "FAIL",
}
summary = pd.DataFrame({"check": list(checks), "status": list(checks.values())})
print(summary.to_string(index=False))
print(f"\nRUN_ID = {RUN_ID}   DEVICE = {DEVICE}   BASE_DIR = {BASE_DIR}")
assert "FAIL" not in checks.values(), "Phase 1 has failing checks — fix them before continuing."
print("\n[OK] PHASE 1 COMPLETE — report these outputs before moving to Phase 2.")

**Interpretation.** Phase 1 is complete when no row is `FAIL`. Please report back:
1. The output of **1.2** (GPU name & memory),
2. The library versions from **1.4**,
3. The summary table of **1.8**,
4. Any warning or error message.

➡️ Continue with **PHASE 2** below.

---
# PHASE 2 — BugInPy Download

> **If your Colab session was restarted** (the variables are gone), first re-run Phase 1:
> 1.1 → 1.2 → 1.3 → 1.4 → 1.5 → 1.6 → 1.7 (set `RESUME_RUN_ID` to your previous run ID in 1.7) → then continue here.

**Goal of the phase:** obtain the official BugInPy benchmark, pin its exact version, keep an untouched archived copy, and verify that every file is accessible.

### Important scientific note: what BugInPy actually contains
BugInPy (official repository: [`soarsmu/BugsInPy`](https://github.com/soarsmu/BugsInPy), Widyasari et al., ESEC/FSE 2020) is a **benchmark of metadata**, not a source-code archive. For each project and each bug it stores:

| File | Content |
|---|---|
| `projects/<project>/project.info` | GitHub URL of the real project |
| `projects/<project>/bugs/<id>/bug.info` | `buggy_commit_id`, `fixed_commit_id`, `test_file`, `python_version` |
| `projects/<project>/bugs/<id>/bug_patch.txt` | the developer's fix as a unified *diff* (buggy → fixed) |
| `projects/<project>/bugs/<id>/run_test.sh` | the test command(s) that expose the bug |
| `requirements.txt`, `setup.sh` | environment needed to run the tests |

The actual **buggy and fixed source code** is obtained later (Phase 4) by checking out the two commits of each original project. Phase 2 therefore downloads only the benchmark itself (~13 MB).

Sections:
* 2.1 Clone the official repository and record its exact commit
* 2.2 Preserve the original dataset (archive + checksums)
* 2.3 Inspect the directory structure
* 2.4 Verify that all files are accessible and well-formed
* 2.5 Record the dataset version and Phase 2 summary

## 2.1 Clone the official BugInPy repository

**Explanation.** We `git clone` the official repository onto the fast local disk (`/content/work/BugsInPy`) and record the exact commit hash, date and message of the version obtained.

**Scientific objective.** The repository is still maintained, so "BugInPy" is not a fixed object: different download dates can give different bug sets. The commit hash is the **dataset version** that must be reported in the thesis. To reproduce a previous run exactly, set `BUGSINPY_COMMIT` to that hash.

**Expected output.** A commit hash (40 hex characters), its date, and `Working tree clean: True`. (When this notebook was prepared, the latest commit was `11c5f1ee…` from 9 Oct 2025. Yours may be newer if the repository has been updated since.)

In [ ]:
# ---- 2.1 Clone BugsInPy ----
BUGSINPY_URL    = CONFIG["dataset"]["repository"] + ".git"
BUGSINPY_COMMIT = None      # None = latest; or pin a hash, e.g. "11c5f1eea954a42132cfd06bf257766a7963e0fd"
BUGSINPY_DIR    = PATHS["work"] / "BugsInPy"

if not (BUGSINPY_DIR / ".git").exists():
    print(f"Cloning {BUGSINPY_URL} ...")
    t0 = time.time()
    git.Repo.clone_from(BUGSINPY_URL, BUGSINPY_DIR)
    print(f"Cloned in {time.time() - t0:.1f}s")
else:
    print("Repository already present locally — reusing it.")

bip_repo = git.Repo(BUGSINPY_DIR)
if BUGSINPY_COMMIT:
    bip_repo.git.checkout(BUGSINPY_COMMIT)

head = bip_repo.head.commit
BUGSINPY_VERSION = {
    "url": BUGSINPY_URL,
    "commit": head.hexsha,
    "commit_date": head.committed_datetime.isoformat(),
    "commit_message": head.message.strip().splitlines()[0],
    "downloaded_at": datetime.datetime.now().isoformat(),
}
for k, v in BUGSINPY_VERSION.items():
    print(f"{k:<15}: {v}")
print("Working tree clean:", not bip_repo.is_dirty(untracked_files=True))

# ---- Verification ----
assert re.fullmatch(r"[0-9a-f]{40}", head.hexsha)
assert (BUGSINPY_DIR / "projects").is_dir(), "'projects/' folder not found in the clone."
assert not bip_repo.is_dirty(untracked_files=True), "Local clone was modified!"
print("\n[OK] 2.1 passed")

**Interpretation.** The printed `commit` is the dataset version of this experiment; it is written into `CONFIG` in 2.5. A clean working tree means the local copy is identical to the official version.

## 2.2 Preserve the original dataset

**Explanation.** We store an untouched copy of the benchmark in `data/raw/` on Drive:
* `BugsInPy_<commit>.tar.gz`: a compressed archive of the full clone (including the git history);
* `BugsInPy_<commit>_manifest.csv`: path, size and **SHA-256** of every file;
* `BugsInPy_<commit>_source.json`: where and when the data came from, plus the archive checksum.

Existing archives are never overwritten. Later phases only **read** from the local clone, and the manifest lets us prove at any time that no original file was modified.

**Scientific objective.** Data provenance and integrity: the raw data is kept exactly as published and can be verified by checksum.

**Expected output.** Archive path and size (≈ a few MB), the number of files in the manifest (≈ 2,000+), and `[OK] 2.2 passed`.

In [ ]:
# ---- 2.2 Preserve original dataset ----
import tarfile

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def build_manifest(root):
    rows = []
    for p in sorted(Path(root).rglob("*")):
        if p.is_file() and ".git" not in p.relative_to(root).parts:
            rows.append({"path": p.relative_to(root).as_posix(),
                         "size_bytes": p.stat().st_size,
                         "sha256": sha256_file(p)})
    return pd.DataFrame(rows)

tag           = BUGSINPY_VERSION["commit"][:12]
archive_path  = PATHS["raw"] / f"BugsInPy_{tag}.tar.gz"
manifest_path = PATHS["raw"] / f"BugsInPy_{tag}_manifest.csv"
source_path   = PATHS["raw"] / f"BugsInPy_{tag}_source.json"

if archive_path.exists():
    print("Archive already exists — NOT overwritten:", archive_path)
else:
    tmp = PATHS["work"] / archive_path.name          # build locally, then copy (faster than writing to Drive)
    with tarfile.open(tmp, "w:gz") as tar:
        tar.add(BUGSINPY_DIR, arcname="BugsInPy")
    shutil.copy2(tmp, archive_path); tmp.unlink()
    print("Archive created:", archive_path)

manifest_df = build_manifest(BUGSINPY_DIR)
if not manifest_path.exists():
    manifest_df.to_csv(manifest_path, index=False)

archive_sha = sha256_file(archive_path)
if not source_path.exists():
    save_json({**BUGSINPY_VERSION, "archive": archive_path.name, "archive_sha256": archive_sha,
               "n_files": len(manifest_df), "total_bytes": int(manifest_df.size_bytes.sum())}, source_path)

print(f"Archive size   : {archive_path.stat().st_size / 1024**2:.2f} MB")
print(f"Archive SHA-256: {archive_sha}")
print(f"Files in manifest (excluding .git): {len(manifest_df)}  "
      f"({manifest_df.size_bytes.sum() / 1024**2:.2f} MB)")

# ---- Verification: archive is readable and complete; local files match the stored manifest ----
with tarfile.open(archive_path, "r:gz") as tar:
    n_arch = sum(1 for m in tar.getmembers() if m.isfile() and "/.git/" not in m.name)
assert n_arch == len(manifest_df), f"Archive has {n_arch} files, manifest {len(manifest_df)}"
stored = pd.read_csv(manifest_path)
assert stored.equals(manifest_df), "Local clone differs from the stored manifest!"
print("\n[OK] 2.2 passed")

**Interpretation.** The raw data is now preserved in Drive and protected by checksums. If you re-run this cell later, it will **not** overwrite the archive. It will, however, check that the local files still match the manifest saved the first time.

## 2.3 Directory structure

**Explanation.** We display the top-level layout, the list of projects with their number of bug folders, and the full content of one example bug.

**Scientific objective.** Understand the raw format before writing any parser, and detect unexpected files.

**Expected output.**
* Top level: `Dockerfile`, `README.md`, `framework/`, `projects/` …
* A table of projects (≈ 17) with bug counts.
* Any **unexpected entries**. For example, when this notebook was prepared there was a stray Office lock file `projects/pandas/bugs/~$Bugs info.xlsx`. It is not a bug folder and will be ignored.
* The content of `black` bug 1: `project.info`, `bug.info` and the first lines of `bug_patch.txt`.

In [ ]:
# ---- 2.3 Directory structure ----
PROJECTS_DIR = BUGSINPY_DIR / "projects"

print("Top level of the repository:")
for p in sorted(BUGSINPY_DIR.iterdir()):
    if p.name != ".git":
        print("  ", p.name + ("/" if p.is_dir() else ""))

rows, unexpected = [], []
for proj in sorted(PROJECTS_DIR.iterdir(), key=lambda p: p.name.lower()):
    if not proj.is_dir():
        unexpected.append(proj.relative_to(BUGSINPY_DIR).as_posix()); continue
    bug_dirs = [b for b in (proj / "bugs").iterdir() if b.is_dir()] if (proj / "bugs").is_dir() else []
    unexpected += [b.relative_to(BUGSINPY_DIR).as_posix()
                   for b in (proj / "bugs").iterdir() if not b.is_dir()] if (proj / "bugs").is_dir() else []
    rows.append({"project": proj.name, "n_bug_folders": len(bug_dirs),
                 "has_project_info": (proj / "project.info").is_file(),
                 "other_files": ", ".join(sorted(f.name for f in proj.iterdir()
                                                 if f.name not in ("bugs", "project.info")))})
structure_df = pd.DataFrame(rows)
print(f"\nProjects found: {len(structure_df)}")
display(structure_df)
print("Unexpected (non-folder) entries:", unexpected or "none")

# Example bug
ex = PROJECTS_DIR / "black" / "bugs" / "1"
if not ex.is_dir():
    ex = next(b for b in (PROJECTS_DIR / structure_df.project[0] / "bugs").iterdir() if b.is_dir())
print(f"\n=== Example: {ex.relative_to(BUGSINPY_DIR)} ===")
print("Files:", sorted(f.name for f in ex.iterdir()))
print("\n--- project.info ---\n" + (ex.parent.parent / "project.info").read_text().strip())
print("\n--- bug.info ---\n" + (ex / "bug.info").read_text().strip())
print("\n--- bug_patch.txt (first 25 lines) ---")
print("\n".join((ex / "bug_patch.txt").read_text(errors="replace").splitlines()[:25]))

# ---- Verification ----
assert len(structure_df) > 0, "No projects found."
assert structure_df.has_project_info.all(), "Some projects have no project.info"
print("\n[OK] 2.3 passed")

**Interpretation.** The diff example shows the label source we will use in Phase 4: lines starting with `-` exist only in the **buggy** version, and lines starting with `+` exist only in the **fixed** version. Unexpected entries are recorded and skipped; they are not deleted, because the raw data must stay untouched.

## 2.4 Verify accessibility and well-formedness of every bug

**Explanation.** For **every** bug folder we check that the required files exist and can be read, then parse `project.info` and `bug.info` (format: `key="value"`) into a table called the *bug inventory*.

Checks:
* `bug.info`, `bug_patch.txt` and `run_test.sh` exist;
* `buggy_commit_id` and `fixed_commit_id` are present and are valid hexadecimal git hashes;
* the patch is non-empty, readable and modifies at least one `.py` file.

**Scientific objective.** Detect broken or atypical entries *before* they silently corrupt the dataset. Anomalies are reported, not hidden.

**Note — short commit hashes.** When this notebook was prepared, the `pandas` bugs used **abbreviated** buggy commit IDs (7 characters, e.g. `45fee32`) instead of full 40-character hashes. These are still valid git references, and Phase 4 will expand them to full hashes when checking out the code. They are therefore flagged as `short_buggy_sha` (a note, not an error).

**Expected output.** An anomaly table and a summary: number of bugs found, number fully valid, and number with notes. The inventory is saved to `data/interim/bugsinpy_inventory.csv`.

In [ ]:
# ---- 2.4 Accessibility & well-formedness ----
KV = re.compile(r'(\w+)\s*=\s*"([^"]*)"')
HEX = re.compile(r"[0-9a-f]{7,40}")

def parse_info(path):
    return dict(KV.findall(Path(path).read_text(encoding="utf-8", errors="replace")))

records, anomalies = [], []
for proj in sorted(p for p in PROJECTS_DIR.iterdir() if p.is_dir()):
    pinfo = parse_info(proj / "project.info") if (proj / "project.info").is_file() else {}
    for bdir in sorted((b for b in (proj / "bugs").iterdir() if b.is_dir()), key=lambda b: int(b.name)):
        rec = {"project": proj.name, "bug_id": int(bdir.name),
               "sample_bug_key": f"{proj.name}-{bdir.name}",
               "github_url": pinfo.get("github_url")}
        issues = []
        for f in ("bug.info", "bug_patch.txt", "run_test.sh"):
            rec[f"has_{f.split('.')[0]}"] = (bdir / f).is_file()
            if not rec[f"has_{f.split('.')[0]}"]:
                issues.append(f"missing_{f}")
        info = parse_info(bdir / "bug.info") if rec["has_bug"] else {}
        for k in ("buggy_commit_id", "fixed_commit_id", "test_file", "python_version", "pythonpath"):
            rec[k] = info.get(k)
        for k in ("buggy_commit_id", "fixed_commit_id"):
            v = (rec[k] or "").strip()
            if not HEX.fullmatch(v):
                issues.append(f"invalid_{k}")
            elif len(v) < 40:
                issues.append(f"short_{k.split('_')[0]}_sha")
        if rec["has_bug_patch"]:
            try:
                patch = (bdir / "bug_patch.txt").read_text(encoding="utf-8")
            except UnicodeDecodeError:
                patch = (bdir / "bug_patch.txt").read_text(encoding="latin-1"); issues.append("patch_not_utf8")
            files_changed = re.findall(r"^diff --git a/(\S+) b/", patch, flags=re.M)
            rec["patch_lines"] = len(patch.splitlines())
            rec["patch_files_changed"] = len(files_changed)
            rec["patch_py_files_changed"] = sum(f.endswith(".py") for f in files_changed)
            if not patch.strip():                 issues.append("empty_patch")
            elif rec["patch_py_files_changed"] == 0: issues.append("no_py_file_in_patch")
        rec["has_setup_sh"] = (bdir / "setup.sh").is_file()
        rec["has_requirements"] = (bdir / "requirements.txt").is_file()
        rec["issues"] = ";".join(issues)
        records.append(rec)

inventory_df = pd.DataFrame(records)
NOTE_ONLY = {"short_buggy_sha", "short_fixed_sha"}
inventory_df["is_valid"] = inventory_df.issues.apply(
    lambda s: all(i in NOTE_ONLY for i in s.split(";") if i))

issue_counts = Counter(i for s in inventory_df.issues for i in s.split(";") if i)
print("Issue / note counts:", dict(issue_counts) or "none")
print("\nIssues per project:")
display(inventory_df[inventory_df.issues != ""].groupby("project").issues
        .agg(lambda s: dict(Counter(i for x in s for i in x.split(";")))).to_frame())

inv_path = PATHS["interim"] / "bugsinpy_inventory.csv"
inventory_df.to_csv(inv_path, index=False)
print(f"\nBugs found          : {len(inventory_df)}")
print(f"Valid (incl. notes) : {inventory_df.is_valid.sum()}")
print(f"With blocking issues: {(~inventory_df.is_valid).sum()}")
print("Saved:", inv_path)
display(inventory_df.head())

# ---- Verification ----
assert inventory_df.sample_bug_key.is_unique, "Duplicate bug keys!"
assert inventory_df.has_bug.all() and inventory_df.has_bug_patch.all(), "Some bugs miss bug.info / bug_patch.txt"
assert inventory_df.is_valid.mean() > 0.95, "More than 5% of bugs are malformed — investigate before continuing."
print("\n[OK] 2.4 passed")

**Interpretation.** `sample_bug_key` (e.g. `black-1`) uniquely identifies each bug instance and will be carried through every later phase, which is necessary to prevent bug-instance leakage in Phase 6. Bugs with blocking issues are **not deleted**; they will be excluded in Phase 4, and the exclusion will be reported. `patch_py_files_changed` already shows that some fixes touch several files, which matters for the labeling strategy.

## 2.5 Record the dataset version and Phase 2 summary

**Explanation.** The exact dataset version (commit, date, archive checksum) is written into `CONFIG` and the config file of this run is updated.

**Expected output.** The updated `dataset` section of `CONFIG` and a summary table with `PASS` on every row.

In [ ]:
# ---- 2.5 Record dataset version & summary ----
CONFIG["dataset"].update({
    "commit": BUGSINPY_VERSION["commit"],
    "commit_date": BUGSINPY_VERSION["commit_date"],
    "downloaded_at": BUGSINPY_VERSION["downloaded_at"],
    "raw_archive": str(archive_path),
    "raw_archive_sha256": archive_sha,
    "manifest": str(manifest_path),
    "inventory": str(inv_path),
    "n_projects": int(inventory_df.project.nunique()),
    "n_bugs": int(len(inventory_df)),
})
cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
print(json.dumps(CONFIG["dataset"], indent=2))

# integrity re-check: originals untouched
still_same = build_manifest(BUGSINPY_DIR).equals(pd.read_csv(manifest_path))
checks2 = {
    "Repository cloned (commit recorded)":  "PASS" if CONFIG["dataset"]["commit"] else "FAIL",
    "Raw archive preserved on storage":     "PASS" if archive_path.exists() else "FAIL",
    "Manifest + checksums saved":           "PASS" if manifest_path.exists() else "FAIL",
    "Original files unmodified":            "PASS" if still_same else "FAIL",
    "All bug folders readable":             "PASS" if inventory_df.has_bug.all() else "FAIL",
    f"Valid bugs: {inventory_df.is_valid.sum()}/{len(inventory_df)}": "PASS",
    "Config updated with dataset version":  "PASS" if json.loads(cfg_path.read_text())["dataset"]["commit"] else "FAIL",
}
print()
print(pd.DataFrame({"check": list(checks2), "status": list(checks2.values())}).to_string(index=False))
assert "FAIL" not in checks2.values()
print(f"\n[OK] PHASE 2 COMPLETE — dataset version {CONFIG['dataset']['commit'][:12]} "
      f"({CONFIG['dataset']['n_projects']} projects, {CONFIG['dataset']['n_bugs']} bugs).")

**Interpretation.** The dataset is downloaded, versioned, preserved and verified. Please report back:
1. The output of **2.1** (commit hash and date),
2. The project table of **2.3** and any unexpected entries,
3. The issue counts and summary of **2.4**,
4. The summary table of **2.5**.

➡️ Continue with **PHASE 3** below.

---
# PHASE 3 — Dataset Exploration

> **After a Colab restart:** re-run Phase 1 (with `RESUME_RUN_ID` set in 1.7), then 2.1 → 2.5. Phase 2 does not overwrite anything that is already saved.

**Goal of the phase:** describe BugInPy quantitatively **before** designing the classification dataset. Each statistic here informs a design decision in Phases 4–6.

| Question | Why it matters later |
|---|---|
| How many bugs per project? | Class balance and project-level splitting are feasible only if projects are not too skewed (Phase 6). |
| How many distinct buggy / fixed versions (commits)? | Number of code checkouts needed in Phase 4. |
| Which Python files do the fixes touch? Source or tests? | Which files the code samples are extracted from (Phase 4). |
| How large are the fixes (lines, hunks, files)? | Whether a bug can be localised to one function (the labeling granularity). |
| Which tests exist? | Available evidence of the bug. |
| Do bugs share commits? | **Label reliability**: a "fixed" version may still contain *another* bug (Phase 4). |

**Scope note.** BugInPy stores *patches*, not full source trees (see Phase 2). In this phase, "Python source files" therefore means the files **modified by the developer fixes**. The total number of Python files in each project version will be measured in Phase 4, when the code is checked out.

Sections:
* 3.1 Projects, bug instances, buggy and fixed versions
* 3.2 Parsing the bug-fix patches
* 3.3 Python source files touched by fixes
* 3.4 Bug-fix characteristics (size and granularity)
* 3.5 Available tests and execution metadata
* 3.6 Commit relationships between bugs (label-reliability preview)
* 3.7 Saving the statistics and Phase 3 summary

## 3.1 Projects, bug instances, buggy and fixed versions

**Explanation.** From the inventory built in 2.4, we count per project: the number of bug instances, the number of **distinct buggy commits** (buggy versions) and the number of **distinct fixed commits** (fixed versions). Commits are compared on their first 7 hex characters, because the pandas entries use abbreviated hashes.

**Scientific objective.** Measure dataset size and **skew across projects**. A heavy imbalance (one project dominating) directly threatens a project-level 70/15/15 split.

**Expected output.** A per-project table, totals (with commit `11c5f1ee` you should get 17 projects and 501 bugs), the share of the largest project, and a bar chart saved to `figures/phase3_bugs_per_project.png`.

In [ ]:
# ---- 3.1 Projects / bugs / versions ----
# Reload from disk so Phase 3 depends only on saved Phase 2 outputs.
inventory_df = pd.read_csv(PATHS["interim"] / "bugsinpy_inventory.csv", keep_default_na=False,
                           dtype={"buggy_commit_id": str, "fixed_commit_id": str, "issues": str})
inventory_df["buggy7"] = inventory_df.buggy_commit_id.str.strip().str[:7]
inventory_df["fixed7"] = inventory_df.fixed_commit_id.str.strip().str[:7]

COLOR_1, COLOR_2 = "#2a78d6", "#eb6834"   # series colours (blue, orange)
sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e6e6", "axes.edgecolor": "#bbbbbb"})

def save_fig(fig, name):
    p = PATHS["figures"] / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight")
    print("Figure saved:", p)

proj_df = (inventory_df.groupby("project")
           .agg(n_bugs=("bug_id", "size"),
                n_buggy_versions=("buggy7", "nunique"),
                n_fixed_versions=("fixed7", "nunique"),
                n_valid=("is_valid", "sum"))
           .sort_values("n_bugs", ascending=False))
proj_df["share_%"] = (100 * proj_df.n_bugs / proj_df.n_bugs.sum()).round(1)
display(proj_df)

totals = {
    "projects": int(inventory_df.project.nunique()),
    "bug_instances": int(len(inventory_df)),
    "valid_bug_instances": int(inventory_df.is_valid.sum()),
    "distinct_buggy_versions": int(proj_df.n_buggy_versions.sum()),
    "distinct_fixed_versions": int(proj_df.n_fixed_versions.sum()),
    "largest_project": proj_df.index[0],
    "largest_project_share_%": float(proj_df["share_%"].iloc[0]),
    "median_bugs_per_project": float(proj_df.n_bugs.median()),
}
print(pd.Series(totals).to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(proj_df.index[::-1], proj_df.n_bugs[::-1], color=COLOR_1, height=0.7)
for y, v in enumerate(proj_df.n_bugs[::-1]):
    ax.text(v + 1, y, str(v), va="center", fontsize=8, color="#333333")
ax.set_xlabel("Number of bug instances"); ax.set_ylabel("")
ax.set_title("BugInPy — bug instances per project", loc="left")
ax.grid(axis="y", visible=False)
save_fig(fig, "phase3_bugs_per_project"); plt.show()

# ---- Verification ----
assert proj_df.n_bugs.sum() == len(inventory_df)
assert (proj_df.n_buggy_versions <= proj_df.n_bugs).all() and (proj_df.n_fixed_versions <= proj_df.n_bugs).all()
print("\n[OK] 3.1 passed")

**Interpretation (to be confirmed with your output).** Look at `largest_project_share_%`. If one project (e.g. pandas) holds about a third of all bugs, then whichever split receives that project will dominate the data. This is the main reason a plain 70/15/15 project split may be unreliable (Phase 6). When `n_buggy_versions < n_bugs` for a project, several bugs share the same buggy commit (analysed in 3.6).

## 3.2 Parsing the bug-fix patches

**Explanation.** Each `bug_patch.txt` is a unified diff. We parse it into one row per **(bug, modified file)**, recording:
* file path, and whether it is a Python file and/or a test file;
* new / deleted / renamed file flags;
* number of hunks (`@@ … @@` blocks), added lines (`+`) and removed lines (`-`);
* whether the change touches only **comments or blank lines** (an important case for label noise).

Only lines *inside* hunks are counted, so the `--- a/` and `+++ b/` headers are never miscounted.

**Scientific objective.** Turn the bug-fix information into measurable variables.

**Expected output.** The number of parsed (bug, file) rows, a preview table, and `[OK] 3.2 passed`.

In [ ]:
# ---- 3.2 Patch parser ----
DIFF_HDR = re.compile(r"^diff --git a/(.+?) b/(.+)$")
HUNK_HDR = re.compile(r"^@@ -\d+(?:,\d+)? \+\d+(?:,\d+)? @@(.*)$")
TEST_PAT = re.compile(r"(?:^|/)(?:tests?|testing)(?:/|$)|(?:^|/)test_[^/]*\.py$|_tests?\.py$|(?:^|/)conftest\.py$")

def is_trivial(line_body: str) -> bool:
    s = line_body.strip()
    return s == "" or s.startswith("#")

def parse_patch(text: str):
    files, cur, in_hunk = [], None, False
    for line in text.splitlines():
        m = DIFF_HDR.match(line)
        if m:
            cur = {"file_path": m.group(2), "old_path": m.group(1), "new_file": False, "deleted_file": False,
                   "n_hunks": 0, "lines_added": 0, "lines_removed": 0, "nontrivial_changes": 0,
                   "hunk_contexts": []}
            files.append(cur); in_hunk = False; continue
        if cur is None:
            continue
        if not in_hunk:
            if line.startswith("new file mode"):     cur["new_file"] = True
            elif line.startswith("deleted file mode"): cur["deleted_file"] = True
        h = HUNK_HDR.match(line)
        if h:
            in_hunk = True; cur["n_hunks"] += 1
            cur["hunk_contexts"].append(h.group(1).strip()); continue
        if in_hunk and line[:1] in "+-":
            key = "lines_added" if line[0] == "+" else "lines_removed"
            cur[key] += 1
            cur["nontrivial_changes"] += not is_trivial(line[1:])
    return files

rows = []
for r in tqdm(inventory_df.itertuples(), total=len(inventory_df), desc="parsing patches"):
    p = PROJECTS_DIR / r.project / "bugs" / str(r.bug_id) / "bug_patch.txt"
    text = p.read_text(encoding="utf-8", errors="replace") if p.is_file() else ""
    for f in parse_patch(text):
        f.update(project=r.project, bug_id=r.bug_id, sample_bug_key=r.sample_bug_key)
        rows.append(f)

patch_files_df = pd.DataFrame(rows)
patch_files_df["renamed"]      = patch_files_df.file_path != patch_files_df.old_path
patch_files_df["is_python"]    = patch_files_df.file_path.str.endswith(".py")
patch_files_df["is_test_file"] = patch_files_df.file_path.str.contains(TEST_PAT)
patch_files_df["lines_changed"] = patch_files_df.lines_added + patch_files_df.lines_removed
patch_files_df["comment_or_blank_only"] = (patch_files_df.lines_changed > 0) & (patch_files_df.nontrivial_changes == 0)
patch_files_df["hunk_contexts"] = patch_files_df.hunk_contexts.apply(lambda l: " || ".join(l))

cols = ["sample_bug_key", "file_path", "is_python", "is_test_file", "new_file", "deleted_file",
        "n_hunks", "lines_added", "lines_removed", "comment_or_blank_only"]
print(f"Parsed (bug, file) rows: {len(patch_files_df)}  from {patch_files_df.sample_bug_key.nunique()} bugs")
display(patch_files_df[cols].head(8))

# ---- Verification: cross-check against the simple counts made in 2.4 ----
chk = patch_files_df.groupby("sample_bug_key").size()
ref = inventory_df.set_index("sample_bug_key").patch_files_changed
ref = ref[ref > 0]
assert chk.reindex(ref.index).fillna(0).astype(int).equals(ref.astype(int)), "File counts disagree with 2.4"
assert (patch_files_df[["n_hunks", "lines_added", "lines_removed"]] >= 0).all().all()
print("\n[OK] 3.2 passed (file counts agree with Phase 2 inventory)")

**Interpretation.** `hunk_contexts` contains the text git prints after `@@`. It is often the enclosing `def`/`class`, but **not reliably**: git's default heuristic only recognises unindented lines, so methods inside classes are frequently attributed to the class or to an earlier function. The exact enclosing function will be resolved with Python's `ast` module in Phase 4. Here it is only a hint.

## 3.3 Python source files touched by fixes

**Explanation.** We classify the modified files as Python source / Python test / non-Python, and count per bug how many **Python source files** the fix touches.

**Scientific objective.** Code samples for the classifier should come from **non-test Python source files**. Bugs whose fix touches no such file cannot yield a buggy code sample. Bugs whose fix touches several files are harder to localise.

**Expected output.** A file-type table, the distribution of "source .py files per bug", the list of bugs with no source `.py` change, and a figure.

In [ ]:
# ---- 3.3 Python files touched ----
def ftype(r):
    if not r.is_python: return "non-Python"
    return "Python test" if r.is_test_file else "Python source"
patch_files_df["file_type"] = patch_files_df.apply(ftype, axis=1)

type_tbl = patch_files_df.groupby("file_type").agg(n_file_changes=("file_path", "size"),
                                                    n_bugs=("sample_bug_key", "nunique"))
display(type_tbl)

src_py = patch_files_df[patch_files_df.file_type == "Python source"]
per_bug_src = (src_py.groupby("sample_bug_key").size()
               .reindex(inventory_df.sample_bug_key, fill_value=0).rename("n_src_py_files"))
inventory_df = inventory_df.merge(per_bug_src, left_on="sample_bug_key", right_index=True, how="left")

dist = per_bug_src.value_counts().sort_index()
print("Source .py files modified per bug:")
print(dist.to_string())
print(f"\nDistinct Python source files touched by fixes: {src_py[['project','file_path']].drop_duplicates().shape[0]}")
print(f"Bugs touching exactly ONE source .py file    : {(per_bug_src == 1).sum()} "
      f"({100 * (per_bug_src == 1).mean():.1f}%)")
no_src = inventory_df[inventory_df.n_src_py_files == 0][["sample_bug_key", "patch_files_changed", "issues"]]
print(f"Bugs with NO source .py change               : {len(no_src)}")
display(no_src)
print("Non-Python files changed (top 10 extensions):",
      dict(Counter(Path(f).suffix or "<none>" for f in
                   patch_files_df.loc[~patch_files_df.is_python, "file_path"]).most_common(10)) or "none")

fig, ax = plt.subplots(figsize=(7, 3.5))
d = dist.copy(); d.index = [str(i) if i < 5 else "5+" for i in d.index]; d = d.groupby(level=0, sort=False).sum()
ax.bar(d.index, d.values, color=COLOR_1, width=0.6)
for x, v in zip(d.index, d.values): ax.text(x, v + 3, str(v), ha="center", fontsize=8)
ax.set_xlabel("Python source files modified by the fix"); ax.set_ylabel("Number of bugs")
ax.set_title("Fix spread across source files", loc="left"); ax.grid(axis="x", visible=False)
save_fig(fig, "phase3_src_files_per_bug"); plt.show()

# ---- Verification ----
assert per_bug_src.sum() == len(src_py)
print("\n[OK] 3.3 passed")

**Interpretation.** Most bugs should change one source file. Bugs listed with **no source `.py` change** (for example the keras bug with an empty patch, or fixes that only touch configuration/test files) cannot provide a buggy sample and will be excluded in Phase 4, with the exclusion documented. If the fixes also touch test files, those tests were added or updated by the developer together with the fix. Test files will never be used as code samples.

## 3.4 Bug-fix characteristics (size and granularity)

**Explanation.** For each bug, restricted to **Python source files**, we compute the hunks, added lines, removed lines and total changed lines. We then group the fixes into categories:
* **single-line**: exactly 1 changed line (e.g. one `-` and/or one `+`, counted as ≤ 2 changed lines with ≤ 1 removed);
* **single-hunk**: one contiguous block;
* **single-file / multi-file**.

We also flag fixes that are **addition-only** (no line removed). In these cases the buggy code is *missing* code, so the buggy version contains no "wrong" line to point at.

**Scientific objective.** The labeling granularity (function level, Phase 4) is reliable only if fixes are small and localised. Addition-only and comment-only fixes are known sources of **label noise**.

**Expected output.** A descriptive statistics table (mean / median / quartiles / max), the category counts, per-project medians and a histogram.

In [ ]:
# ---- 3.4 Fix size ----
fix = (src_py.groupby("sample_bug_key")
       .agg(n_hunks=("n_hunks", "sum"), lines_added=("lines_added", "sum"),
            lines_removed=("lines_removed", "sum"), nontrivial=("nontrivial_changes", "sum"),
            n_files=("file_path", "size"))
       .reset_index())
fix["lines_changed"] = fix.lines_added + fix.lines_removed
fix = fix.merge(inventory_df[["sample_bug_key", "project"]], on="sample_bug_key")

print("Descriptive statistics (Python source files only, per bug):")
display(fix[["n_files", "n_hunks", "lines_added", "lines_removed", "lines_changed"]]
        .describe(percentiles=[.25, .5, .75, .9]).round(2))

cats = {
    "single-line fix (≤1 removed, ≤2 changed)": ((fix.lines_removed <= 1) & (fix.lines_changed <= 2)).sum(),
    "single-hunk fix":                         (fix.n_hunks == 1).sum(),
    "single-file fix":                         (fix.n_files == 1).sum(),
    "multi-file fix":                          (fix.n_files > 1).sum(),
    "addition-only fix (0 lines removed)":     (fix.lines_removed == 0).sum(),
    "comment/blank-only fix":                  (fix.nontrivial == 0).sum(),
    "large fix (> 50 changed lines)":          (fix.lines_changed > 50).sum(),
}
cat_df = pd.DataFrame({"n_bugs": cats}); cat_df["% of bugs with src fix"] = (100 * cat_df.n_bugs / len(fix)).round(1)
display(cat_df)

print("Per-project median fix size:")
display(fix.groupby("project")[["n_hunks", "lines_changed"]].median().sort_values("lines_changed"))

fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.logspace(0, np.log10(max(fix.lines_changed.max(), 10)), 25)
ax.hist(fix.lines_changed.clip(lower=1), bins=bins, color=COLOR_1, edgecolor="white", linewidth=1)
ax.set_xscale("log"); ax.set_xlabel("Changed lines in Python source files (log scale)")
ax.set_ylabel("Number of bugs"); ax.set_title("Distribution of fix size", loc="left")
ax.axvline(fix.lines_changed.median(), color="#333333", lw=1, ls="--")
ax.text(fix.lines_changed.median() * 1.08, ax.get_ylim()[1] * 0.92,
        f"median = {fix.lines_changed.median():.0f}", fontsize=8)
save_fig(fig, "phase3_fix_size_hist"); plt.show()

# ---- Verification ----
assert len(fix) == (inventory_df.n_src_py_files > 0).sum()
assert (fix.lines_changed >= 0).all()
print("\n[OK] 3.4 passed")

**Interpretation.** A small median and a high share of single-hunk fixes would support function-level labeling: the buggy behaviour is concentrated in one function. **Multi-file and large fixes** may mix the real correction with refactoring, which creates *noisy* labels (some modified functions are not actually buggy). **Addition-only fixes** mean the bug is an *omission*: the buggy function looks syntactically fine and is detectable only from what is missing. These categories will be kept as metadata so that they can be analysed separately in Phase 10.

## 3.5 Available tests and execution metadata

**Explanation.** We analyse the bug-revealing tests in `bug.info` (`test_file`, possibly several separated by `;`) and `run_test.sh`, together with the declared Python version and environment files.

**Scientific objective.** Tests are the evidence that the buggy version fails and the fixed version passes. They are not used as classifier input in this experiment. However, they matter for describing the benchmark and for later PhD experiments (e.g. test-based repair validation).

**Expected output.** Test-file counts per bug, the number of test commands, Python version distribution, availability of `setup.sh` and `requirements.txt`, and three example test commands.

In [ ]:
# ---- 3.5 Tests ----
def read_lines(p):
    p = Path(p)
    return [l for l in p.read_text(errors="replace").splitlines() if l.strip() and not l.startswith("#")] if p.is_file() else []

inventory_df["test_files"] = inventory_df.test_file.fillna("").apply(
    lambda s: [t.strip() for t in str(s).split(";") if t.strip()])
inventory_df["n_test_files"] = inventory_df.test_files.str.len()
inventory_df["n_test_commands"] = [len(read_lines(PROJECTS_DIR / r.project / "bugs" / str(r.bug_id) / "run_test.sh"))
                                   for r in inventory_df.itertuples()]

test_summary = {
    "bugs with >=1 test file declared": int((inventory_df.n_test_files > 0).sum()),
    "bugs with >=1 test command":       int((inventory_df.n_test_commands > 0).sum()),
    "distinct test files":              int(len({(r.project, t) for r in inventory_df.itertuples() for t in r.test_files})),
    "mean test commands per bug":       round(float(inventory_df.n_test_commands.mean()), 2),
    "max test commands per bug":        int(inventory_df.n_test_commands.max()),
    "bugs with setup.sh":               int(inventory_df.has_setup_sh.sum()),
    "bugs with requirements.txt":       int(inventory_df.has_requirements.sum()),
}
print(pd.Series(test_summary).to_string())
print("\nTest files declared per bug:", inventory_df.n_test_files.value_counts().sort_index().to_dict())
print("Declared Python versions   :", inventory_df.python_version.value_counts().to_dict())

print("\nPer-project test availability:")
display(inventory_df.groupby("project").agg(bugs=("bug_id", "size"),
                                            mean_test_cmds=("n_test_commands", "mean"),
                                            with_setup_sh=("has_setup_sh", "sum")).round(2))

print("Example test commands:")
for r in inventory_df.sample(3, random_state=SEED).itertuples():
    cmds = read_lines(PROJECTS_DIR / r.project / "bugs" / str(r.bug_id) / "run_test.sh")
    print(f"  {r.sample_bug_key:<14} {cmds[0][:110] if cmds else '<none>'}")

# ---- Verification ----
assert (inventory_df.n_test_commands >= 0).all()
print("\n[OK] 3.5 passed")

**Interpretation.** Nearly every bug should have at least one test command. This confirms that BugInPy bugs are **test-validated**: the buggy version fails the test and the fixed version passes it. Note, however, that a passing test proves only that *this* bug is fixed. It does **not** prove that the fixed code is free of other bugs. This leads directly to 3.6.

## 3.6 Commit relationships between bugs (label-reliability preview)

**Explanation.** Within each project we look for:
1. **Shared buggy commit**: several bugs have the *same* buggy version.
2. **Shared fixed commit**: several bugs are fixed by the *same* commit.
3. **Chained bugs**: the **fixed commit of bug A is the buggy commit of bug B**. The "fixed" code of A is officially *buggy* for B.
4. **Overlapping functions (approximate)**: two bugs of the same project modify the same function, identified by file and git hunk context (`def …`).

**Scientific objective.** This is the empirical basis of the labeling strategy (Phase 4). It shows why **a function from a fixed version cannot automatically be treated as a clean, independent negative**. It also shows which bugs must stay in the **same split** to avoid bug-instance leakage (Phase 6).

**Expected output.** Counts for each relationship type and example groups.

In [ ]:
# ---- 3.6 Commit relationships ----
inv = inventory_df[["project", "sample_bug_key", "buggy7", "fixed7"]]

def groups(col):
    g = inv.groupby(["project", col]).sample_bug_key.apply(list)
    return g[g.str.len() > 1]

shared_buggy, shared_fixed = groups("buggy7"), groups("fixed7")
chains = (inv.merge(inv, left_on=["project", "fixed7"], right_on=["project", "buggy7"], suffixes=("_A", "_B"))
             .query("sample_bug_key_A != sample_bug_key_B"))

# same function (file + git hunk context) modified by two different bugs of the same project.
# APPROXIMATE: git hunk contexts are heuristic (see 3.2); exact AST-based matching comes in Phase 4.
ctx = (src_py.assign(ctx=src_py.hunk_contexts.str.split(r" \|\| "))
             .explode("ctx")
             .loc[lambda d: d.ctx.str.match(r"(?:async\s+)?def\s")]   # function contexts only
             .groupby(["project", "file_path", "ctx"]).sample_bug_key.unique())
overlap = ctx[ctx.str.len() > 1]
bugs_in_overlap = sorted({b for l in overlap for b in l})

rel = {
    "groups sharing a buggy commit":            len(shared_buggy),
    "bugs in shared-buggy groups":              int(shared_buggy.apply(len).sum()),
    "groups sharing a fixed commit":            len(shared_fixed),
    "bugs in shared-fixed groups":              int(shared_fixed.apply(len).sum()),
    "chains fixed(A) == buggy(B)":              len(chains),
    "functions modified by >1 bug (approx.)":   len(overlap),
    "bugs touching such a function (approx.)":  len(bugs_in_overlap),
}
for k, v in rel.items(): print(f"{k:<45} {v}")

print("\nExamples — shared buggy commit:");  print(shared_buggy.head(5).to_string())
print("\nExamples — chains (A fixed → becomes B buggy):")
print(chains[["sample_bug_key_A", "sample_bug_key_B", "fixed7_A"]].head(5).to_string(index=False))
print("\nExamples — same function modified by several bugs (approx.):"); print(overlap.head(5).to_string())

# ---- Verification ----
assert all(len(set(l)) == len(l) for l in shared_buggy)
print("\n[OK] 3.6 passed")

**Interpretation (fill in from your output).** Each non-zero count is a concrete threat to naive labeling:
* **Chains / overlapping regions**: a function taken from the *fixed* version of bug A may be the *buggy* function of bug B (or may be modified again later). Labeling it `0` would be wrong.
* **Shared buggy commits**: the same buggy file may appear for several bugs. Without grouping, *identical* code could land in both train and test (**leakage**).

These observations motivate the labeling strategy, which will be **presented and discussed with you before implementation** at the beginning of Phase 4.

## 3.7 Saving statistics and Phase 3 summary

**Explanation.** All exploration tables are saved (CSV and JSON) for the thesis.

**Expected output.** A list of saved files and a summary table with `PASS` on every row.

In [ ]:
# ---- 3.7 Save & summary ----
out = {
    "per_project":   PATHS["metrics"] / "phase3_per_project.csv",
    "patch_files":   PATHS["interim"] / "phase3_patch_files.csv",
    "fix_stats":     PATHS["interim"] / "phase3_fix_stats.csv",
    "inventory_ext": PATHS["interim"] / "bugsinpy_inventory_phase3.csv",
    "summary_json":  PATHS["metrics"] / "phase3_dataset_summary.json",
}
proj_df.to_csv(out["per_project"])
patch_files_df.to_csv(out["patch_files"], index=False)
fix.to_csv(out["fix_stats"], index=False)
inventory_df.assign(test_files=inventory_df.test_files.str.join(";")).to_csv(out["inventory_ext"], index=False)
save_json({"run_id": RUN_ID, "dataset_commit": CONFIG["dataset"]["commit"], "totals": totals,
           "file_types": type_tbl.reset_index().to_dict("records"),
           "fix_categories": {k: int(v) for k, v in cats.items()},
           "fix_size_describe": fix[["n_files", "n_hunks", "lines_changed"]].describe().round(3).to_dict(),
           "tests": test_summary, "commit_relationships": {k: int(v) for k, v in rel.items()}},
          out["summary_json"])
for k, p in out.items(): print(f"{k:<14} -> {p}")

checks3 = {
    "Counts consistent with Phase 2":         "PASS" if totals["bug_instances"] == CONFIG["dataset"]["n_bugs"] else "FAIL",
    "All patches parsed":                     "PASS" if patch_files_df.sample_bug_key.nunique() >= totals["valid_bug_instances"] else "FAIL",
    "Source .py fixes identified":            "PASS" if len(fix) > 0 else "FAIL",
    "Tests analysed":                         "PASS" if (inventory_df.n_test_commands > 0).any() else "FAIL",
    "Commit relationships analysed":          "PASS",
    "Figures saved":                          "PASS" if len(list(PATHS["figures"].glob("phase3_*.png"))) >= 3 else "FAIL",
    "Statistics saved (CSV + JSON)":          "PASS" if all(p.exists() for p in out.values()) else "FAIL",
}
print()
print(pd.DataFrame({"check": list(checks3), "status": list(checks3.values())}).to_string(index=False))
assert "FAIL" not in checks3.values()
print("\n[OK] PHASE 3 COMPLETE")

**Please report back:**
1. The `totals` printed in **3.1** and the bar chart,
2. The file-type table and "bugs with NO source .py change" from **3.3**,
3. The descriptive statistics and fix categories from **3.4**,
4. The relationship counts from **3.6**,
5. The summary table of **3.7**.

➡️ Continue with **PHASE 4** below.

---
# PHASE 4 — Dataset Construction

> **After a Colab restart:** re-run Phase 1 (with `RESUME_RUN_ID` in 1.7) and Phase 2 (2.1–2.5). Phase 4 reloads the Phase 3 results from Drive, so Phase 3 does not need to be re-run. Downloaded source files are cached on Drive and are not downloaded again.

## Approved labeling strategy (Option C)

**Unit of analysis.** One Python **function or method** (`def` / `async def`), extracted with `ast` (abstract syntax tree). Methods are named `Class.method`; nested functions stay inside their parent function. Decorators are included.

**Code source.** For each bug, the files modified by the developer fix are retrieved from the original project at the **buggy commit** and at the **fixed commit** (`git show <commit>:<path>`). Test files are never used.

| Label | Source (`label_source`) | Definition |
|---|---|---|
| **1** | `buggy_modified` | Function in the **buggy** version that overlaps a removed line or an insertion point of the patch **and** whose code is semantically different in the fixed version (normalized AST, docstrings ignored). |
| **0** | `fixed_counterpart` *(hard negative)* | The same function in the **fixed** version. |
| **0** | `unchanged_same_file` *(easy negative)* | **One** function per bug, drawn at random (seed 42) from the same modified file(s), that the fix does not change, **with a length between ½× and 2× the buggy function's length** (closest length if no candidate is in that range). See the decision log below. |

**Reliability filters (based on Phase 3, section 3.6).**
* A function modified by **any other bug** of the same project (chains, functions fixed several times) is **never** used as a negative.
* An "unchanged" negative must not be modified by **any** bug, must be identical in the buggy and fixed versions, and must have at least 3 lines.
* Changes with no semantic effect (only comments, docstrings or formatting) do not create positives.
* Changes outside any function (imports, constants) do not create samples. They are counted and reported.

**Decision log — dataset versions.**
| Version | Easy-negative selection | Status |
|---|---|---|
| `v1_random` | uniform random among eligible unchanged functions | first run; files `phase4_dataset.*` **kept unchanged** for comparison |
| `v2_length_matched` | random among eligible functions with length in [½×, 2×] of the bug's buggy function length (median if several); closest length otherwise | **current dataset**, files `phase4_dataset_v2_length_matched.*` |

*Reason for v2:* in v1 the easy negatives had a median length of 12 lines vs 26 for buggy functions — a potential **shortcut** (a classifier could learn "short ⇒ clean"). A **length-only baseline** will be trained in Phase 7 to verify that the shortcut is reduced.

**Meaning of label 0.** "No known bug in BugInPy", **not** "proven bug-free". This is a threat to validity that will be discussed in Phase 10.

Sections:
* 4.1 Load Phase 3 results and record the labeling configuration
* 4.2 Retrieve the original project repositories
* 4.3 Retrieve the buggy and fixed versions of every modified source file
* 4.4 Function extraction with `ast`
* 4.5 Buggy functions (label 1)
* 4.6 Negative samples (label 0) with reliability filters
* 4.7 Assemble the dataset: identifiers, metadata, leakage groups
* 4.8 Label reliability investigation, saving and summary

## 4.1 Load Phase 3 results and record the labeling configuration

**Explanation.** Phase 4 reloads the saved outputs of Phase 3 (bug inventory, parsed patch files, fix statistics), so it does not depend on variables kept in memory. The labeling parameters are written to `CONFIG["labeling"]`.

**Expected output.** 501 bugs and 647 (bug, file) rows loaded, the labeling configuration, and `[OK] 4.1 passed`.

In [ ]:
# ---- 4.1 Load Phase 3 results & labeling config ----
# Prerequisite check: variables created by earlier cells in THIS Colab session
_required = {"PATHS": "1.6", "SEED": "1.5", "CONFIG": "1.7", "RUN_ID": "1.7", "save_json": "1.7",
             "BUGSINPY_DIR": "2.1", "PROJECTS_DIR": "2.3"}
_missing = sorted({cell for name, cell in _required.items() if name not in globals()})
if _missing:
    raise RuntimeError(f"Run these cells first (in order): {', '.join(_missing)} "
                       f"— then 1.8 and 2.1-2.5. Cause: the Colab session was restarted or a cell was skipped.")
import subprocess, difflib, zlib, copy, tarfile
from concurrent.futures import ThreadPoolExecutor

inv4 = pd.read_csv(PATHS["interim"] / "bugsinpy_inventory_phase3.csv", keep_default_na=False,
                   dtype={"buggy_commit_id": str, "fixed_commit_id": str, "buggy7": str, "fixed7": str})
pf4  = pd.read_csv(PATHS["interim"] / "phase3_patch_files.csv", keep_default_na=False)
fix4 = pd.read_csv(PATHS["interim"] / "phase3_fix_stats.csv")
for c in ["is_python", "is_test_file", "new_file", "deleted_file"]:
    pf4[c] = pf4[c].astype(str).str.lower().eq("true")
inv4["is_valid"] = inv4["is_valid"].astype(str).str.lower().eq("true")

CONFIG["labeling"] = {
    "strategy": "Option C (paired hard negatives + 1 unchanged function per bug), approved by the researcher",
    "unit": "function/method (ast.FunctionDef / AsyncFunctionDef); methods as Class.method; nested functions stay in parent",
    "positive": "buggy-version function overlapping removed lines / insertion points of the patch AND semantically changed (normalized AST, docstrings removed)",
    "negative_fixed_counterpart": True,
    "negatives_unchanged_per_bug": 1,
    "unchanged_min_lines": 3,
    "exclude_test_files": True,
    "exclude_functions_modified_by_other_bugs_from_negatives": True,
    "sampling_seed": SEED,
}
cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
print(json.dumps(CONFIG["labeling"], indent=2))
print(f"\nBugs: {len(inv4)} | (bug,file) rows: {len(pf4)} | bugs with source fix: {len(fix4)}")

# ---- Verification ----
assert len(inv4) == CONFIG["dataset"]["n_bugs"] and len(pf4) > 0
print("\n[OK] 4.1 passed")

## 4.2 Retrieve the original project repositories

**Explanation.** For each of the 17 projects we create a **partial clone** of the original GitHub repository (`--filter=blob:none --no-checkout`). It contains the full commit history but **no file contents**. File contents are downloaded later, and only for the files we need. Then every `buggy_commit_id` and `fixed_commit_id` (including the abbreviated pandas hashes) is resolved to a full 40-character hash.

**Scientific objective.** Obtain the exact code versions referenced by BugInPy, and verify that every commit still exists upstream.

**Expected output.** A table with 17 repositories (clone time and size; ansible, pandas and matplotlib are the largest, around 50–100 MB). Then `Commits resolved: 1002 / 1002`. Total time: about 1–3 minutes.

In [ ]:
# ---- 4.2 Partial clones & commit resolution ----
REPOS_DIR = PATHS["work"] / "repos"; REPOS_DIR.mkdir(parents=True, exist_ok=True)

def git(repo, *args, check=True, text=True, input=None):
    r = subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=text, input=input)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed: {r.stderr if text else r.stderr.decode()[:500]}")
    return r

def clone_project(project, url):
    dest = REPOS_DIR / project
    if (dest / ".git").exists():
        return {"project": project, "status": "cached", "seconds": 0.0}
    t0 = time.time()
    r = subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout", url.rstrip("/"), str(dest)],
                       capture_output=True, text=True)
    return {"project": project, "status": "cloned" if r.returncode == 0 else f"FAILED: {r.stderr[:200]}",
            "seconds": round(time.time() - t0, 1)}

proj_urls = inv4.groupby("project").github_url.first()
t0 = time.time()
with ThreadPoolExecutor(max_workers=6) as ex:
    clone_df = pd.DataFrame(ex.map(lambda kv: clone_project(*kv), proj_urls.items()))
clone_df["url"] = clone_df.project.map(proj_urls)
clone_df["size_MB"] = [round(sum(f.stat().st_size for f in (REPOS_DIR / p).rglob("*") if f.is_file()) / 1024**2, 1)
                       if (REPOS_DIR / p).exists() else None for p in clone_df.project]
display(clone_df)
print(f"Total clone time: {time.time() - t0:.0f}s")

def resolve(project, ref):
    r = git(REPOS_DIR / project, "rev-parse", "--verify", "-q", ref.strip() + "^{commit}", check=False)
    if r.returncode != 0 and re.fullmatch(r"[0-9a-f]{40}", ref.strip()):        # commit not on any branch: fetch it
        git(REPOS_DIR / project, "fetch", "-q", "--filter=blob:none", "origin", ref.strip(), check=False)
        r = git(REPOS_DIR / project, "rev-parse", "--verify", "-q", ref.strip() + "^{commit}", check=False)
    return r.stdout.strip() if r.returncode == 0 else None

inv4["buggy_sha"] = [resolve(p, c) for p, c in zip(inv4.project, inv4.buggy_commit_id)]
inv4["fixed_sha"] = [resolve(p, c) for p, c in zip(inv4.project, inv4.fixed_commit_id)]
n_ok = inv4.buggy_sha.notna().sum() + inv4.fixed_sha.notna().sum()
print(f"Commits resolved: {n_ok} / {2 * len(inv4)}")
unresolved = inv4[inv4.buggy_sha.isna() | inv4.fixed_sha.isna()][["sample_bug_key", "buggy_commit_id", "fixed_commit_id"]]
print("Unresolved:", "none" if unresolved.empty else ""); 
if not unresolved.empty: display(unresolved)

# ---- Verification ----
assert clone_df.status.isin(["cloned", "cached"]).all(), "Some clones failed (see table)."
assert n_ok >= 0.98 * 2 * len(inv4), "Too many unresolved commits."
print("\n[OK] 4.2 passed")

**Interpretation.** A resolved hash proves that the exact code version used by BugInPy is still available upstream, so the dataset can be rebuilt by anyone. The full hashes (not the abbreviated pandas ones) are stored with every sample.

## 4.3 Retrieve the buggy and fixed versions of every modified source file

**Explanation.** For each (bug, modified **source** `.py` file) we need two files: the **buggy** version (at `buggy_sha`, at the old path) and the **fixed** version (at `fixed_sha`). Their contents are downloaded in **batches**, one network request per group of files, because fetching files one by one takes about 3 s each.

The files are cached locally and archived to Drive (`data/interim/phase4_source_files.tar.gz`), so they are never downloaded twice.

**Consistency check (important).** For every patch we verify that each **removed** line (`-`) appears **exactly at its line number in the buggy file**, and each **added** line (`+`) in the fixed file. This proves that the patch describes exactly the difference between the two retrieved versions, so line-based labeling is valid.

**Expected output.** About 1,200 files retrieved (a few minutes the first time). Then the number of fully consistent (bug, file) pairs: with commit `11c5f1ee`, **594 / 599** in our test run, with 100% exact line matches inside them. Finally, the list of inconsistent pairs. In our test run these were 3 very large pandas refactorings of `datetimelike.py` (≈ 1,500 removed lines, whose added side does not match the fixed commit), plus pandas-5 and one scrapy-33 file.

In [ ]:
# ---- 4.3 Retrieve source files (batched) & patch consistency ----
SRC_CACHE = PATHS["work"] / "src_cache"
SRC_ARCHIVE = PATHS["interim"] / "phase4_source_files.tar.gz"
if SRC_ARCHIVE.exists() and not SRC_CACHE.exists():
    print("Restoring source files from Drive archive ...")
    with tarfile.open(SRC_ARCHIVE) as tar: tar.extractall(PATHS["work"])

def cache_path(project, sha, path): return SRC_CACHE / project / sha[:12] / path

ok_bugs = inv4[inv4.is_valid & inv4.buggy_sha.notna() & inv4.fixed_sha.notna()].set_index("sample_bug_key")
src_files = pf4[(pf4.file_type == "Python source") & pf4.sample_bug_key.isin(ok_bugs.index)].copy()
src_files["buggy_sha"] = src_files.sample_bug_key.map(ok_bugs.buggy_sha)
src_files["fixed_sha"] = src_files.sample_bug_key.map(ok_bugs.fixed_sha)

needed = set()
for r in src_files.itertuples():
    if not r.new_file:     needed.add((r.project, r.buggy_sha, r.old_path))
    if not r.deleted_file: needed.add((r.project, r.fixed_sha, r.file_path))
todo = sorted(x for x in needed if not cache_path(*x).exists())
print(f"Files needed: {len(needed)} | already cached: {len(needed) - len(todo)} | to download: {len(todo)}")

fetch_log = []
for project, grp in tqdm(pd.DataFrame(todo, columns=["project", "sha", "path"]).groupby("project"), desc="projects"):
    repo = REPOS_DIR / project
    oid_of = {}
    for sha, g in grp.groupby("sha"):                      # 1) blob ids from trees (no download needed)
        out = git(repo, "ls-tree", "-r", "-z", sha, "--", *g.path.tolist()).stdout
        for entry in filter(None, out.split("\0")):
            meta, p = entry.split("\t", 1); oid_of[(sha, p)] = meta.split()[2]
    oids = sorted(set(oid_of.values()))
    for i in range(0, len(oids), 300):                     # 2) one batched download per 300 blobs
        git(repo, "-c", "fetch.negotiationAlgorithm=noop", "fetch", "-q", "--no-tags",
            "--no-write-fetch-head", "--filter=blob:none", "origin", *oids[i:i + 300], check=False)
    for r in grp.itertuples():                             # 3) read blobs locally and cache
        oid = oid_of.get((r.sha, r.path))
        res = git(repo, "cat-file", "blob", oid, check=False, text=False) if oid else None
        ok = res is not None and res.returncode == 0
        if ok:
            dst = cache_path(project, r.sha, r.path); dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(res.stdout)
        fetch_log.append({"project": project, "sha": r.sha, "path": r.path, "ok": ok})

missing = [x for x in needed if not cache_path(*x).exists()]
print(f"Retrieved: {len(needed) - len(missing)} / {len(needed)} files")

if todo:   # archive the cache to Drive (overwrites only the cache archive, never raw data)
    tmp = PATHS["work"] / "phase4_source_files.tar.gz"
    with tarfile.open(tmp, "w:gz") as tar: tar.add(SRC_CACHE, arcname="src_cache")
    shutil.copy2(tmp, SRC_ARCHIVE); tmp.unlink()
    print("Source-file cache archived to:", SRC_ARCHIVE)

# ---- Patch line-level parser (line numbers) ----
HUNK_RE  = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
DIFF_RE  = re.compile(r"^diff --git a/(.+?) b/(.+)$")
def parse_patch_lines(text):
    files, cur, o, n = {}, None, None, None
    for line in text.splitlines():
        m = DIFF_RE.match(line)
        if m:
            cur = files.setdefault(m.group(2), {"removed": [], "added": [], "insert_at": []}); o = None; continue
        if cur is None: continue
        h = HUNK_RE.match(line)
        if h:
            o, n = int(h.group(1)), int(h.group(3))
            if h.group(2) == "0": o += 1          # pure insertion: old range is empty
            if h.group(4) == "0": n += 1          # pure deletion: new range is empty
            continue
        if o is None or line.startswith("\\"): continue
        if line.startswith("-"):   cur["removed"].append((o, line[1:])); o += 1
        elif line.startswith("+"): cur["added"].append((n, line[1:])); cur["insert_at"].append(o); n += 1
        else:                      o += 1; n += 1
    return files

def read_src(project, sha, path):
    p = cache_path(project, sha, path)
    return p.read_bytes().decode("utf-8", errors="replace").replace("\r\n", "\n").replace("\r", "\n") if p.exists() else None

PATCH_LINES, cons = {}, []
for key in tqdm(src_files.sample_bug_key.unique(), desc="consistency"):
    b = ok_bugs.loc[key]
    PATCH_LINES[key] = parse_patch_lines((PROJECTS_DIR / b.project / "bugs" / str(b.bug_id) / "bug_patch.txt")
                                         .read_text(encoding="utf-8", errors="replace"))
for r in src_files.itertuples():
    pl = PATCH_LINES[r.sample_bug_key].get(r.file_path, {"removed": [], "added": []})
    bl = (read_src(r.project, r.buggy_sha, r.old_path) or "").split("\n")
    fl = (read_src(r.project, r.fixed_sha, r.file_path) or "").split("\n")
    rm_ok = sum(1 for ln, t in pl["removed"] if 0 < ln <= len(bl) and bl[ln - 1].rstrip() == t.rstrip())
    ad_ok = sum(1 for ln, t in pl["added"]   if 0 < ln <= len(fl) and fl[ln - 1].rstrip() == t.rstrip())
    cons.append({"sample_bug_key": r.sample_bug_key, "file_path": r.file_path,
                 "removed": len(pl["removed"]), "removed_ok": rm_ok, "added": len(pl["added"]), "added_ok": ad_ok})
cons_df = pd.DataFrame(cons)
cons_df["consistent"] = (cons_df.removed == cons_df.removed_ok) & (cons_df.added == cons_df.added_ok)
src_files = src_files.merge(cons_df[["sample_bug_key", "file_path", "consistent"]], on=["sample_bug_key", "file_path"])

print(f"\n(bug, file) pairs fully consistent: {cons_df.consistent.sum()} / {len(cons_df)} "
      f"({100 * cons_df.consistent.mean():.1f}%)")
cc = cons_df[cons_df.consistent]
print(f"  in consistent pairs: removed lines {cc.removed_ok.sum()}/{cc.removed.sum()}, "
      f"added lines {cc.added_ok.sum()}/{cc.added.sum()} match exactly")
ic = cons_df[~cons_df.consistent]
print(f"  in inconsistent pairs: removed {ic.removed_ok.sum()}/{ic.removed.sum()}, added {ic.added_ok.sum()}/{ic.added.sum()}")
incons = cons_df[~cons_df.consistent]
if len(incons): print("Inconsistent pairs (excluded from labeling):"); display(incons)

# ---- Verification ----
assert len(missing) <= 0.02 * len(needed), f"{len(missing)} files could not be retrieved"
assert cons_df.consistent.mean() > 0.95, "Patches do not match retrieved files — investigate."
print("\n[OK] 4.3 passed")

**Interpretation.** A consistency close to 100% shows that the BugInPy patch is exactly the difference between the two commits we retrieved. The line numbers of the patch can therefore be mapped safely onto functions. Inconsistent pairs, if any, are excluded rather than guessed.

## 4.4 Function extraction with `ast`

**Explanation.** `extract_functions(source)` parses a Python file and returns every function and method with:
* `qualname` (e.g. `Parser.parse_args`). Duplicate names in the same scope, such as `@property` and its `@x.setter`, get a suffix (`name#2`);
* `start_line` / `end_line` (including decorators);
* `code`: the source text of the function, dedented;
* `ast_hash`: a hash of the **normalized AST** (docstrings removed, positions ignored). Two versions with the same `ast_hash` are semantically identical, even if comments, formatting or docstrings differ.

Functions inside `if` / `try` blocks at module or class level are included (common in compatibility code).

**Expected output.** The functions found in one example file, a demonstration that comments and docstrings do not change `ast_hash`, and the number of files that could **not** be parsed by the Colab Python version.

In [ ]:
# ---- 4.4 Function extraction ----
def _strip_docstrings(node):
    for n in ast.walk(node):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.body \
           and isinstance(n.body[0], ast.Expr) and isinstance(getattr(n.body[0], "value", None), ast.Constant) \
           and isinstance(n.body[0].value.value, str):
            n.body = n.body[1:] or [ast.Pass()]
    return node

def ast_hash(node):
    return hashlib.md5(ast.dump(_strip_docstrings(copy.deepcopy(node)), include_attributes=False).encode()).hexdigest()

def extract_functions(source):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                    # old code: invalid escape sequences etc.
        tree = ast.parse(source)
    lines, out, seen = source.split("\n"), [], Counter()
    def visit(body, prefix):
        for node in body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                q = prefix + node.name; seen[q] += 1
                start = min([node.lineno] + [d.lineno for d in node.decorator_list])
                out.append({"qualname": q if seen[q] == 1 else f"{q}#{seen[q]}", "name": node.name,
                            "start_line": start, "end_line": node.end_lineno,
                            "n_lines": node.end_lineno - start + 1,
                            "code": textwrap.dedent("\n".join(lines[start - 1:node.end_lineno])),
                            "ast_hash": ast_hash(node)})
            elif isinstance(node, ast.ClassDef):
                visit(node.body, prefix + node.name + ".")
            else:                                          # if / try / with / for at module or class level
                for f in ("body", "orelse", "finalbody"):
                    visit(getattr(node, f, []) or [], prefix)
                for h in getattr(node, "handlers", []) or []:
                    visit(h.body, prefix)
    visit(tree.body, "")
    return out

# Demonstration
demo = '''
class A:
    @property
    def x(self):
        'doc'
        return self._x  # comment
    @x.setter
    def x(self, v):
        self._x = v
def f(a):
    return a + 1
'''
demo2 = demo.replace("'doc'", "'another docstring'").replace("# comment", "")
d1, d2 = extract_functions(demo), extract_functions(demo2)
display(pd.DataFrame(d1)[["qualname", "start_line", "end_line", "n_lines", "ast_hash"]])
print("Docstring/comment change keeps ast_hash:", d1[0]["ast_hash"] == d2[0]["ast_hash"])
print("Different code gives different ast_hash:",
      extract_functions("def f(a):\n    return a + 1")[0]["ast_hash"] != extract_functions("def f(a):\n    return a - 1")[0]["ast_hash"])

# Parse every retrieved file once (cached)
FUNCS, parse_fail = {}, []
for (project, sha, path) in tqdm(sorted(needed), desc="parsing files"):
    s = read_src(project, sha, path)
    if s is None: continue
    try:
        FUNCS[(project, sha, path)] = extract_functions(s)
    except (SyntaxError, ValueError, RecursionError) as e:
        parse_fail.append({"project": project, "sha": sha[:12], "path": path, "error": f"{type(e).__name__}: {e}"[:120]})
n_funcs = sum(len(v) for v in FUNCS.values())
print(f"\nFiles parsed: {len(FUNCS)} / {len(needed)} | functions extracted: {n_funcs}")
print("Parse failures:", len(parse_fail))
if parse_fail: display(pd.DataFrame(parse_fail))

# ---- Verification ----
assert d1[0]["qualname"] == "A.x" and d1[1]["qualname"] == "A.x#2" and d1[2]["qualname"] == "f"
assert d1[0]["ast_hash"] == d2[0]["ast_hash"]
assert len(FUNCS) >= 0.95 * len(needed), "Too many files failed to parse."
print("\n[OK] 4.4 passed")

**Interpretation.** A parse failure means the Colab Python version cannot read that file. This happens, for example, with very old syntax such as `async` used as a variable name before Python 3.7. These files are excluded and reported, never patched by hand.

## 4.5 Buggy functions (label 1)

**Explanation.** For each (bug, source file):
1. Take the line numbers of the removed lines and of the insertion points from the patch (buggy-file coordinates).
2. **Candidate** functions are the buggy-version functions whose line span contains one of these positions. An insertion just after the last line of a function counts as an insertion into that function.
3. A candidate becomes a **positive** if its `ast_hash` differs in the fixed version (same `qualname`), or if the fix removed or renamed it.
4. A candidate with an identical `ast_hash` was changed only in comments, docstrings or formatting. It is **not** a positive (`non_semantic`).
5. Changed lines outside every function are counted as `module_level` changes.

**Expected output.** The number of positives, the number of bugs with at least one positive, the distribution of positives per bug, and the reasons why some bugs have no positive.

In [ ]:
# ---- 4.5 Positives ----
pos_rows, bug_status, cand_stats = [], {}, Counter()
for key in tqdm(ok_bugs.index, desc="labeling bugs"):
    b = ok_bugs.loc[key]
    files = src_files[src_files.sample_bug_key == key]
    if files.empty:
        bug_status[key] = "no_source_py_change"; continue
    reasons = []
    for r in files.itertuples():
        if not r.consistent:                     reasons.append("patch_inconsistent"); continue
        if r.new_file:                           reasons.append("new_file_only"); continue
        bf = FUNCS.get((b.project, b.buggy_sha, r.old_path))
        ff = FUNCS.get((b.project, b.fixed_sha, r.file_path), []) if not r.deleted_file else []
        if bf is None:                           reasons.append("parse_or_fetch_failure"); continue
        pl = PATCH_LINES[key].get(r.file_path, {"removed": [], "insert_at": []})
        positions = [ln for ln, _ in pl["removed"]] + list(pl["insert_at"])
        fixed_by_q = {f["qualname"]: f for f in ff}
        hit = set()
        for pos in positions:
            inside = [f for f in bf if f["start_line"] <= pos <= f["end_line"] + 1]
            if not inside: cand_stats["module_level_positions"] += 1
            for f in inside: hit.add(f["qualname"])
        for f in bf:
            if f["qualname"] not in hit: continue
            cf = fixed_by_q.get(f["qualname"])
            if cf is not None and cf["ast_hash"] == f["ast_hash"]:
                cand_stats["non_semantic_candidates"] += 1; continue
            pos_rows.append({"sample_bug_key": key, "project": b.project, "bug_id": int(b.bug_id),
                             "file_path": r.file_path, "old_path": r.old_path,
                             "buggy_sha": b.buggy_sha, "fixed_sha": b.fixed_sha, **f,
                             "has_fixed_counterpart": cf is not None,
                             "fixed": cf})
        if not hit: reasons.append("module_level_only")
    if key not in bug_status:   # overwritten below by "has_positive" if the bug produced a positive
        bug_status[key] = Counter(reasons).most_common(1)[0][0] if reasons else "no_semantic_function_change"

positives = pd.DataFrame(pos_rows)
status = pd.Series(bug_status, name="status")
status[status.index.isin(positives.sample_bug_key)] = "has_positive"
for k in inv4.sample_bug_key:
    if k not in status.index: status[k] = "invalid_or_unresolved"

print(f"Positive functions (label 1): {len(positives)}")
print(f"Bugs with >= 1 positive     : {positives.sample_bug_key.nunique()} / {len(inv4)}")
print(f"Positives with fixed counterpart: {positives.has_fixed_counterpart.sum()}")
print("Positives per bug:", positives.groupby("sample_bug_key").size().value_counts().sort_index().to_dict())
print("Candidate statistics:", dict(cand_stats))
print("\nBug status:"); display(status.value_counts().to_frame("n_bugs"))

# ---- Verification ----
assert positives.sample_bug_key.nunique() == (status == "has_positive").sum()
assert not positives.duplicated(["sample_bug_key", "file_path", "qualname"]).any()
print("\n[OK] 4.5 passed")

**Interpretation.** Most bugs should yield exactly one positive function. Bugs without a positive fall into documented categories:
* `module_level_only`: the fix changes imports or constants, not a function;
* `no_semantic_function_change`: only comments or formatting change;
* `parse_or_fetch_failure`, `patch_inconsistent`, `invalid_or_unresolved` (e.g. keras-12).

These bugs are excluded from the classification dataset, and their number is reported in the thesis.

## 4.6 Negative samples (label 0) with reliability filters

**Explanation.**
* `MODIFIED_BY`: for every function key `(project, file, qualname)`, the set of bugs whose fix modifies it.
* **Hard negative** (`fixed_counterpart`): the fixed version of each positive, **kept only if no other bug modifies this function**. This removes chains and functions fixed several times (Phase 3, section 3.6).
* **Easy negative** (`unchanged_same_file`): **one** function per bug, drawn with a seeded random generator (seed = `SEED` + bug key) among the functions of the modified file(s) that:
  * are unchanged between the buggy and fixed versions (same `ast_hash`);
  * are modified by **no** bug in BugInPy;
  * have at least `unchanged_min_lines` lines;
  * **(v2, length matching)** have a length between **½×** and **2×** the reference length `L` = length of the bug's buggy function (median if the bug has several). If no eligible function is in that range, the eligible function with the **closest length** is taken (ties broken by the same seeded generator).

`UNCHANGED_SELECTION = "random"` reproduces the original v1 dataset exactly.

**Expected output.** The number of hard and easy negatives, the number of hard negatives removed by the reliability filter (with examples), the number of bugs without an eligible easy negative and, for v2, how many easy negatives were matched **in range** vs. by **closest length**.

In [ ]:
# ---- 4.6 Negatives ----
# Researcher decision (Option A): length-matched selection of the unchanged (easy) negative.
UNCHANGED_SELECTION = "length_matched"   # "random" -> v1 (original) | "length_matched" -> v2
LEN_RATIO = (0.5, 2.0)                    # accepted length range relative to the buggy function
DATASET_VERSION = {"random": "v1_random", "length_matched": "v2_length_matched"}[UNCHANGED_SELECTION]
print("Dataset version:", DATASET_VERSION)

MODIFIED_BY = defaultdict(set)
for r in positives.itertuples():
    MODIFIED_BY[(r.project, r.file_path, r.qualname)].add(r.sample_bug_key)
    MODIFIED_BY[(r.project, r.old_path, r.qualname)].add(r.sample_bug_key)

hard, hard_rejected = [], []
for r in positives.itertuples():
    if not r.has_fixed_counterpart: continue
    others = (MODIFIED_BY[(r.project, r.file_path, r.qualname)] | MODIFIED_BY[(r.project, r.old_path, r.qualname)]) - {r.sample_bug_key}
    if others:
        hard_rejected.append({"sample_bug_key": r.sample_bug_key, "function": f"{r.file_path}::{r.qualname}",
                              "also_modified_by": ", ".join(sorted(others))}); continue
    f = r.fixed
    hard.append({"sample_bug_key": r.sample_bug_key, "project": r.project, "bug_id": r.bug_id,
                 "file_path": r.file_path, "buggy_sha": r.buggy_sha, "fixed_sha": r.fixed_sha, **f})

UNCH_MIN = CONFIG["labeling"]["unchanged_min_lines"]
easy, no_easy = [], []
for key, grp in positives.groupby("sample_bug_key"):
    b = ok_bugs.loc[key]; pool = []
    for r in src_files[(src_files.sample_bug_key == key) & src_files.consistent & ~src_files.new_file].itertuples():
        bf = FUNCS.get((b.project, b.buggy_sha, r.old_path)) or []
        fixed_hash = {f["qualname"]: f["ast_hash"] for f in FUNCS.get((b.project, b.fixed_sha, r.file_path), [])}
        for f in bf:
            k1, k2 = (b.project, r.file_path, f["qualname"]), (b.project, r.old_path, f["qualname"])
            if MODIFIED_BY.get(k1) or MODIFIED_BY.get(k2):       continue   # modified by some bug
            if fixed_hash.get(f["qualname"]) != f["ast_hash"]:   continue   # not identical in fixed version
            if f["n_lines"] < UNCH_MIN:                          continue
            pool.append((r.file_path, f))
    if not pool:
        no_easy.append(key); continue
    rng = np.random.default_rng([SEED, zlib.crc32(key.encode())])
    ref_len = float(grp.n_lines.median())                       # length of the bug's buggy function(s)
    if UNCHANGED_SELECTION == "random":
        choices, match = pool, "random"
    else:
        lo, hi = LEN_RATIO[0] * ref_len, LEN_RATIO[1] * ref_len
        choices = [x for x in pool if lo <= x[1]["n_lines"] <= hi]
        match = "in_range"
        if not choices:                                         # fallback: closest length
            best = min(abs(x[1]["n_lines"] - ref_len) for x in pool)
            choices = [x for x in pool if abs(x[1]["n_lines"] - ref_len) == best]
            match = "closest"
    fp, f = choices[int(rng.integers(len(choices)))]
    easy.append({"sample_bug_key": key, "project": b.project, "bug_id": int(b.bug_id), "file_path": fp,
                 "buggy_sha": b.buggy_sha, "fixed_sha": b.fixed_sha, "pool_size": len(pool),
                 "n_choices": len(choices), "ref_len": ref_len, "match_type": match, **f})

hard_df, easy_df = pd.DataFrame(hard), pd.DataFrame(easy)
print(f"Hard negatives (fixed counterparts)  : {len(hard_df)}")
print(f"  rejected — modified by another bug : {len(hard_rejected)}")
print(f"  no counterpart (function removed)  : {(~positives.has_fixed_counterpart).sum()}")
print(f"Easy negatives (unchanged, 1 per bug): {len(easy_df)}")
print(f"  bugs with no eligible unchanged fn : {len(no_easy)}")
print(f"  median candidate pool size         : {easy_df.pool_size.median() if len(easy_df) else 0}")
print(f"  selection                          : {UNCHANGED_SELECTION} -> {easy_df.match_type.value_counts().to_dict()}")
if UNCHANGED_SELECTION == "length_matched":
    ratio = easy_df.n_lines / easy_df.ref_len
    print(f"  length ratio easy/buggy (median)   : {ratio.median():.2f}  "
          f"(in [{LEN_RATIO[0]}, {LEN_RATIO[1]}]: {ratio.between(*LEN_RATIO).mean() * 100:.1f}%)")
if hard_rejected:
    print("\nExamples of hard negatives rejected by the reliability filter:")
    display(pd.DataFrame(hard_rejected).head(10))

# ---- Verification ----
for r in easy_df.itertuples():
    assert not MODIFIED_BY.get((r.project, r.file_path, r.qualname)), "easy negative is a modified function!"
for r in hard_df.itertuples():
    assert MODIFIED_BY[(r.project, r.file_path, r.qualname)] <= {r.sample_bug_key}
if UNCHANGED_SELECTION == "length_matched":
    assert (easy_df[easy_df.match_type == "in_range"].n_lines / easy_df[easy_df.match_type == "in_range"].ref_len
            ).between(*LEN_RATIO).all(), "in-range easy negative outside the length range!"
print("\n[OK] 4.6 passed")

**Interpretation.** Each rejected hard negative is a function that **would have been mislabeled `0`** by a naive strategy: its "fixed" version is still modified by another BugInPy bug. Their number quantifies the label noise that our filter removes. Bugs without an eligible easy negative are typically small files where every function was touched by some fix.

## 4.7 Assemble the dataset: identifiers, metadata and leakage groups

**Explanation.** Positives, hard negatives and easy negatives are merged into a single table, with one row per sample:

| Field | Meaning |
|---|---|
| `sample_id` | unique id `bugkey__version__file::qualname` |
| `project`, `bug_id`, `sample_bug_key` | origin of the sample |
| `file_path`, `qualname`, `start_line`, `end_line` | location in the source file |
| `version`, `commit_sha` | `buggy` or `fixed` commit, with the full 40-character hash |
| `label`, `label_source` | 1 / 0 and why |
| `code` | function source code |
| flags | `addition_only_fix`, `single_line_fix`, `large_fix`, `multi_file_fix`, `in_chain`, `shared_buggy_commit`, `multi_bug_function` |
| `leakage_group` | bugs that must stay in the same split |

**Leakage groups.** Bugs of the same project are joined (union-find) if they share a buggy commit, form a chain (fixed(A) = buggy(B)), or modify the same function. All their samples will be kept in the same split in Phase 6.

**Expected output.** Class distribution, counts per `label_source`, the number of leakage groups and the largest groups, and one example of a buggy / fixed pair shown as a diff.

In [ ]:
# ---- 4.7 Assemble dataset ----
cols = ["sample_bug_key", "project", "bug_id", "file_path", "qualname", "name", "start_line", "end_line",
        "n_lines", "code", "ast_hash"]
p = positives.assign(version="buggy", commit_sha=positives.buggy_sha, label=1, label_source="buggy_modified")
h = hard_df.assign(version="fixed", commit_sha=hard_df.fixed_sha, label=0, label_source="fixed_counterpart")
e = easy_df.assign(version="buggy", commit_sha=easy_df.buggy_sha, label=0, label_source="unchanged_same_file")
extra = ["version", "commit_sha", "label", "label_source"]
dataset = pd.concat([p[cols + extra], h[cols + extra], e[cols + extra]], ignore_index=True)
dataset.insert(0, "sample_id", dataset.sample_bug_key + "__" + dataset.version + "__" + dataset.file_path + "::" + dataset.qualname)
dataset.loc[dataset.label_source == "unchanged_same_file", "sample_id"] += "__unchanged"
dataset["dataset_version"] = DATASET_VERSION
_em = easy_df.set_index("sample_bug_key")
_is_e = dataset.label_source == "unchanged_same_file"
dataset["easy_match_type"] = np.where(_is_e, dataset.sample_bug_key.map(_em.match_type), "")
dataset["easy_ref_len"] = np.where(_is_e, dataset.sample_bug_key.map(_em.ref_len), np.nan)

# bug-level reliability flags (from Phase 3 statistics)
fx = fix4.set_index("sample_bug_key")
dataset["addition_only_fix"] = dataset.sample_bug_key.map(fx.lines_removed == 0).fillna(False)
dataset["single_line_fix"]   = dataset.sample_bug_key.map((fx.lines_removed <= 1) & (fx.lines_changed <= 2)).fillna(False)
dataset["large_fix"]         = dataset.sample_bug_key.map(fx.lines_changed > 50).fillna(False)
dataset["multi_file_fix"]    = dataset.sample_bug_key.map(fx.n_files > 1).fillna(False)

inv_k = inv4.set_index("sample_bug_key")
chain_pairs = (inv4.merge(inv4, left_on=["project", "fixed7"], right_on=["project", "buggy7"], suffixes=("_A", "_B"))
                   .query("sample_bug_key_A != sample_bug_key_B")[["sample_bug_key_A", "sample_bug_key_B"]])
in_chain = set(chain_pairs.sample_bug_key_A) | set(chain_pairs.sample_bug_key_B)
shared_b = set(inv4[inv4.duplicated(["project", "buggy7"], keep=False)].sample_bug_key)
dataset["in_chain"] = dataset.sample_bug_key.isin(in_chain)
dataset["shared_buggy_commit"] = dataset.sample_bug_key.isin(shared_b)
dataset["multi_bug_function"] = [len(MODIFIED_BY.get((pr, fp, q), ())) > 1
                                 for pr, fp, q in zip(dataset.project, dataset.file_path, dataset.qualname)]

# leakage groups (union-find over bugs)
parent = {k: k for k in dataset.sample_bug_key.unique()}
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    if a in parent and b in parent: parent[find(a)] = find(b)
for a, b2 in chain_pairs.itertuples(index=False): union(a, b2)
for _, g in inv4[inv4.sample_bug_key.isin(parent)].groupby(["project", "buggy7"]):
    ks = g.sample_bug_key.tolist(); [union(ks[0], k) for k in ks[1:]]
for bugs in MODIFIED_BY.values():
    ks = sorted(bugs); [union(ks[0], k) for k in ks[1:]]
roots = {k: find(k) for k in parent}
gid = {r: f"G{i:04d}" for i, r in enumerate(sorted(set(roots.values())))}
dataset["leakage_group"] = dataset.project + "::" + dataset.sample_bug_key.map(lambda k: gid[roots[k]])

print("Class distribution:"); display(dataset.label.value_counts().rename({1: "buggy (1)", 0: "clean (0)"}).to_frame("n"))
print("By label_source:");    display(dataset.label_source.value_counts().to_frame("n"))
gs = dataset.groupby("leakage_group").sample_bug_key.nunique()
print(f"Leakage groups: {len(gs)} | groups with >1 bug: {(gs > 1).sum()} | largest groups:")
print(gs.sort_values(ascending=False).head(5).to_string())

# Example: one buggy / fixed pair as a diff
ex_key = "black-1" if "black-1" in set(hard_df.sample_bug_key) else hard_df.sample_bug_key.iloc[0]
pair = dataset[(dataset.sample_bug_key == ex_key) & dataset.label_source.isin(["buggy_modified", "fixed_counterpart"])]
q0 = pair.qualname.iloc[0]
bcode = pair[(pair.label == 1) & (pair.qualname == q0)].code.iloc[0]
fcode = pair[(pair.label == 0) & (pair.qualname == q0)].code.iloc[0]
print(f"\nExample {ex_key} — {pair.file_path.iloc[0]}::{q0}  (buggy → fixed)")
print("\n".join(list(difflib.unified_diff(bcode.splitlines(), fcode.splitlines(), "buggy (label 1)", "fixed (label 0)", lineterm="", n=2))[:40]))

# ---- Verification ----
assert dataset.sample_id.is_unique, "sample_id not unique"
assert set(dataset.label.unique()) <= {0, 1}
assert dataset[["project", "bug_id", "file_path", "commit_sha", "code"]].notna().all().all()
assert dataset.commit_sha.str.fullmatch(r"[0-9a-f]{40}").all()
assert dataset.groupby("sample_bug_key").leakage_group.nunique().max() == 1
print("\n[OK] 4.7 passed")

**Interpretation.** The diff shows the only difference between a positive and its hard negative: the developer's fix. The class distribution is **not** 50/50 by construction: about one positive for two negatives, minus the filtered samples. This imbalance is handled in Phases 8–9 (F1, ROC-AUC, class weights if needed). The leakage groups are the units that Phase 6 must never separate.

## 4.8 Label reliability investigation, saving and summary

**Explanation.** We summarise how reliable each part of the dataset is:
* **Bug funnel**: from 501 BugInPy bugs to the bugs that contribute samples, with the reason for each exclusion;
* the share of samples carrying each **reliability flag**, per label source;
* the **label noise removed** by the filters (hard negatives rejected, non-semantic candidates);
* possible **shortcuts and biases**: function length per label source, positives concentrated in a few very large fixes, and exact duplicate code texts.

Then the dataset is saved, together with a per-bug status file and a JSON summary. **File names depend on the dataset version**: v1 uses `phase4_dataset.*`, and v2 uses `phase4_dataset_v2_length_matched.*`. The v1 files are never overwritten by v2.

**Length shortcut check (v2).** We compare the length distributions of v1 and v2 (table and figure). We also compute a purely **descriptive** separability score: the ROC-AUC obtained by using *shorter length* alone as a "clean" score. 0.5 means that length carries no information about the label. This is not a model evaluation. The proper length-only **baseline** (trained on the train split, evaluated on the test split) is part of Phase 7.

**Expected output.** The funnel table, the flag table, the saved paths, and a summary table with `PASS` on every row.

In [ ]:
# ---- 4.8 Reliability investigation, save, summary ----
funnel = pd.DataFrame({
    "step": ["BugInPy bugs", "valid metadata + commits resolved", "with >=1 source .py change",
             "with >=1 positive function (contribute samples)"],
    "n_bugs": [len(inv4), len(ok_bugs), src_files.sample_bug_key.nunique(), positives.sample_bug_key.nunique()]})
display(funnel)
print("Exclusion reasons:"); display(status[status != "has_positive"].value_counts().to_frame("n_bugs"))

flags = ["addition_only_fix", "single_line_fix", "large_fix", "multi_file_fix", "in_chain",
         "shared_buggy_commit", "multi_bug_function"]
print("Share of samples with each reliability flag (%):")
display((100 * dataset.groupby("label_source")[flags].mean()).round(1).T)

print("Function length (lines) per label source — a large gap is a potential SHORTCUT for the classifier:")
display(dataset.groupby("label_source").n_lines.describe(percentiles=[.5, .9]).round(1))

ppb = dataset[dataset.label == 1].groupby("sample_bug_key").size().sort_values(ascending=False)
print(f"Positives contributed by the 5 largest bugs: {ppb.head(5).sum()} of {ppb.sum()} "
      f"({100 * ppb.head(5).sum() / ppb.sum():.1f}%)")
display(ppb.head(5).to_frame("n_positives").join(fix4.set_index("sample_bug_key")[["lines_changed", "n_files", "n_hunks"]]))

dup = dataset.groupby("code").label.agg(["nunique", "size"])
print(f"Exact code duplicates: {(dup['size'] > 1).sum()} code texts appear more than once; "
      f"{(dup['nunique'] > 1).sum()} with CONFLICTING labels (handled in Phase 5)")

noise = {"hard_negatives_rejected_other_bug": len(hard_rejected),
         "non_semantic_candidates_dropped": int(cand_stats["non_semantic_candidates"]),
         "module_level_positions_ignored": int(cand_stats["module_level_positions"]),
         "bugs_without_easy_negative": len(no_easy)}
print("Label noise removed / ignored:", noise)

# ---- Length shortcut check: v1 (original) vs current version ----
from sklearn.metrics import roc_auc_score
def length_auc(df, sources=None):
    d = df if sources is None else df[df.label_source.isin(sources)]
    return roc_auc_score(d.label, d.n_lines) if d.label.nunique() == 2 else float("nan")   # longer => "buggy" score

v1_path = PATHS["interim"] / "phase4_dataset.jsonl"
versions = {DATASET_VERSION: dataset}
if DATASET_VERSION != "v1_random" and v1_path.exists():
    versions["v1_random"] = pd.read_json(v1_path, lines=True)
len_rows, auc_rows = [], []
for v, df in versions.items():
    for src, g in df.groupby("label_source"):
        len_rows.append({"version": v, "label_source": src, "n": len(g), "median": g.n_lines.median(),
                         "mean": round(g.n_lines.mean(), 1), "p90": g.n_lines.quantile(.9)})
    auc_rows.append({"version": v, "AUC_all_samples": round(length_auc(df), 3),
                     "AUC_buggy_vs_unchanged": round(length_auc(df, ["buggy_modified", "unchanged_same_file"]), 3)})
len_cmp = pd.DataFrame(len_rows).sort_values(["label_source", "version"])
auc_cmp = pd.DataFrame(auc_rows)
print("Length distribution (lines) — v1 vs current:"); display(len_cmp)
print("Descriptive length-separability (ROC-AUC of length alone; 0.5 = no information):"); display(auc_cmp)

fig, axes = plt.subplots(1, len(versions), figsize=(5.2 * len(versions), 3.8), sharey=True, squeeze=False)
order = ["buggy_modified", "fixed_counterpart", "unchanged_same_file"]
for ax, v in zip(axes[0], sorted(versions)):
    df = versions[v]
    ax.boxplot([df[df.label_source == s].n_lines for s in order], showfliers=False, widths=0.5,
               medianprops={"color": "#333333"}, boxprops={"color": COLOR_1 if "COLOR_1" in globals() else "#2a78d6"})
    ax.set_xticks([1, 2, 3], ["buggy (1)", "fixed (0)", "unchanged (0)"])
    ax.set_yscale("log"); ax.set_title(v, loc="left"); ax.grid(axis="x", visible=False)
axes[0][0].set_ylabel("Function length (lines, log scale)")
fig.suptitle("Function length by label source (outliers hidden)", x=0.01, ha="left")
fig.tight_layout()
fig_path = PATHS["figures"] / f"phase4_length_by_source_{DATASET_VERSION}.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight"); print("Figure saved:", fig_path); plt.show()

suffix = "" if DATASET_VERSION == "v1_random" else f"_{DATASET_VERSION}"
out4 = {"jsonl": PATHS["interim"] / f"phase4_dataset{suffix}.jsonl",
        "csv": PATHS["interim"] / f"phase4_dataset{suffix}.csv",
        "bug_status": PATHS["interim"] / "phase4_bug_status.csv",
        "hard_rejected": PATHS["interim"] / "phase4_hard_negatives_rejected.csv",
        "length_check": PATHS["metrics"] / f"phase4_length_check{suffix}.csv",
        "summary": PATHS["metrics"] / f"phase4_summary{suffix}.json"}
len_cmp.merge(auc_cmp, on="version").to_csv(out4["length_check"], index=False)
dataset.to_json(out4["jsonl"], orient="records", lines=True, force_ascii=False)
dataset.to_csv(out4["csv"], index=False)
status.rename_axis("sample_bug_key").to_frame().to_csv(out4["bug_status"])
pd.DataFrame(hard_rejected).to_csv(out4["hard_rejected"], index=False)
summary4 = {"run_id": RUN_ID, "dataset_commit": CONFIG["dataset"]["commit"],
            "n_samples": len(dataset), "n_positive": int((dataset.label == 1).sum()),
            "n_negative": int((dataset.label == 0).sum()),
            "by_label_source": dataset.label_source.value_counts().to_dict(),
            "n_bugs_contributing": int(dataset.sample_bug_key.nunique()),
            "n_projects": int(dataset.project.nunique()),
            "n_leakage_groups": int(dataset.leakage_group.nunique()),
            "funnel": funnel.to_dict("records"), "exclusions": status[status != "has_positive"].value_counts().to_dict(),
            "noise_removed": noise, "patch_consistency": float(cons_df.consistent.mean()),
            "parse_failures": len(parse_fail), "dataset_version": DATASET_VERSION,
            "unchanged_selection": UNCHANGED_SELECTION, "length_ratio_range": list(LEN_RATIO),
            "easy_match_types": easy_df.match_type.value_counts().to_dict(),
            "length_check": len_cmp.to_dict("records"), "length_auc": auc_cmp.to_dict("records")}
save_json(summary4, out4["summary"])
CONFIG["labeling"].update({"unchanged_selection": UNCHANGED_SELECTION, "length_ratio_range": list(LEN_RATIO),
                           "current_dataset_version": DATASET_VERSION, "current_dataset_file": str(out4["jsonl"])})
CONFIG["labeling"].setdefault("results", {})[DATASET_VERSION] = {
    k: summary4[k] for k in ["n_samples", "n_positive", "n_negative", "n_bugs_contributing"]}
CONFIG["labeling"].pop("result", None)
CONFIG["planned_baselines"] = ["tfidf_logistic_regression", "tfidf_random_forest",
                               "length_only_logistic_regression (shortcut check, researcher request)"]
save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
for k, v in out4.items(): print(f"{k:<14} -> {v}")

reloaded = pd.read_json(out4["jsonl"], lines=True)
checks4 = {
    "All commits resolved":                     "PASS" if n_ok == 2 * len(inv4) else "WARN",
    "Source files retrieved":                   "PASS" if not missing else "WARN",
    "Patch/file consistency > 95%":             "PASS" if cons_df.consistent.mean() > 0.95 else "FAIL",
    "Positives found for most bugs (> 80%)":    "PASS" if positives.sample_bug_key.nunique() > 0.8 * len(inv4) else "WARN",
    "Both classes present":                     "PASS" if dataset.label.nunique() == 2 else "FAIL",
    "Unique sample ids":                        "PASS" if dataset.sample_id.is_unique else "FAIL",
    "Negatives pass reliability filters":       "PASS",
    "Dataset saved & reloadable":               "PASS" if len(reloaded) == len(dataset) else "FAIL",
}
print()
print(pd.DataFrame({"check": list(checks4), "status": list(checks4.values())}).to_string(index=False))
assert "FAIL" not in checks4.values()
print(f"\n[OK] PHASE 4 COMPLETE [{DATASET_VERSION}] — {len(dataset)} samples "
      f"({summary4['n_positive']} buggy / {summary4['n_negative']} clean) from {summary4['n_bugs_contributing']} bugs.")

**Please report back** the outputs of **4.2** (commits resolved), **4.3** (consistency), **4.4** (parse failures), **4.5** (positives and bug status), **4.6** (negatives and rejections), **4.7** (class distribution, leakage groups, example diff) and **4.8** (summary).

➡️ Continue with **PHASE 5** below.

---
# PHASE 5 — Data Preprocessing & Tokenization

> **After a Colab restart:** Phase 5 only needs **Phase 1** (cells 1.1 → 1.8, with `RESUME_RUN_ID` set in 1.7). It reads the Phase 4 dataset (v2) and the saved configuration from Drive.

**Goal of the phase:** turn the labelled functions into clean, de-duplicated, length-analysed inputs for CodeBERT.

| Step | Decision | Rationale |
|---|---|---|
| Code unit | one function / method (from Phase 4) | approved strategy |
| **Model input** | function **without comments and docstrings**, in canonical formatting (`ast.unparse`) | (1) labels were defined on the *normalized AST*, which ignores comments and docstrings, so the input contains exactly what the label is based on; (2) fixes often **add explanatory comments** (e.g. black-1), a cue that would let a model recognise *fixed* code without understanding the bug; (3) shorter inputs mean less truncation. The original code is kept in `code_raw`. |
| Invalid samples | removed if the function cannot be parsed | — |
| Duplicates | identical model inputs: if the copies have **conflicting labels**, all are removed; if they have the same label, one copy is kept | prevents label noise and train/test duplicate leakage |
| Tokenizer | `microsoft/codebert-base` (RoBERTa BPE) | matches the model |
| Long functions | `max_length = 512`, **head truncation**, padding to 512 | standard for CodeBERT. We measure how often the bug lies beyond the window. The window is **never** chosen using the bug location, because that would leak the label |

Setting `REMOVE_COMMENTS_DOCSTRINGS = False` in 5.2 reproduces the raw-code variant.

Sections:
* 5.1 Load the Phase 4 dataset (v2) and configuration
* 5.2 Normalization of the model input and removal of invalid samples
* 5.3 Duplicate removal
* 5.4 Code length analysis
* 5.5 CodeBERT tokenizer
* 5.6 Token length analysis
* 5.7 Long sequences: truncation and bug visibility
* 5.8 Tokenization (padding and truncation) and saving
* 5.9 Phase 5 summary

## 5.1 Load the Phase 4 dataset (v2) and configuration

**Explanation.** We restore the dataset-related parts of the saved configuration (`config_<RUN_ID>.json`), then load the **current** dataset version recorded in Phase 4 (`v2_length_matched`).

**Expected output.** `1828` samples (`781` buggy / `1047` clean), version `v2_length_matched`, and `[OK] 5.1 passed`.

In [ ]:
# ---- 5.1 Load dataset v2 & configuration ----
_missing = [n for n in ("PATHS", "CONFIG", "RUN_ID", "SEED", "save_json") if n not in globals()]
if _missing:
    raise RuntimeError(f"Run Phase 1 first (cells 1.1 -> 1.8, with RESUME_RUN_ID in 1.7). Missing: {_missing}")
import ast, difflib, transformers
from sklearn.metrics import roc_auc_score
transformers.logging.set_verbosity_error()

cfg_file = PATHS["logs"] / f"config_{RUN_ID}.json"
saved_cfg = json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
for k in ("dataset", "labeling", "planned_baselines"):
    if k in saved_cfg:
        CONFIG[k] = saved_cfg[k]
data4_path = Path(CONFIG.get("labeling", {}).get("current_dataset_file",
                  PATHS["interim"] / "phase4_dataset_v2_length_matched.jsonl"))
assert data4_path.exists(), f"Phase 4 dataset not found: {data4_path} — run Phase 4 first."

df5 = pd.read_json(data4_path, lines=True, dtype={"commit_sha": str})
print("Loaded:", data4_path.name)
print("Dataset version:", df5.dataset_version.unique().tolist())
print(f"Samples: {len(df5)} | buggy (1): {(df5.label == 1).sum()} | clean (0): {(df5.label == 0).sum()}")
display(df5.label_source.value_counts().to_frame("n"))

# ---- Verification ----
need = {"sample_id", "project", "bug_id", "sample_bug_key", "file_path", "qualname", "version",
        "commit_sha", "label", "label_source", "code", "leakage_group"}
assert need <= set(df5.columns), f"Missing columns: {need - set(df5.columns)}"
assert df5.sample_id.is_unique
print("\n[OK] 5.1 passed")

## 5.2 Normalization of the model input and removal of invalid samples

**Explanation.** `normalize_function(code)`:
1. parses the function. Some methods cannot be dedented cleanly (multi-line strings with less indentation), so they are parsed inside a wrapper class;
2. removes docstrings, including those of nested functions and classes;
3. regenerates the code with `ast.unparse`. Comments disappear, and formatting becomes canonical (4-space indentation, uniform quotes).

Samples that still cannot be parsed are **invalid** and removed.

We also measure the **comment cue**: in how many buggy/fixed pairs does the fixed version contain **more comment lines** than the buggy one? This is the shortcut that the normalization removes.

**Expected output.** The number of invalid samples (≈ 0), a before/after example, the comment-cue statistic, and `[OK] 5.2 passed`.

In [ ]:
# ---- 5.2 Normalization & invalid samples ----
REMOVE_COMMENTS_DOCSTRINGS = True

def _strip_docstrings(node):
    for n in ast.walk(node):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.body \
           and isinstance(n.body[0], ast.Expr) and isinstance(getattr(n.body[0], "value", None), ast.Constant) \
           and isinstance(n.body[0].value.value, str):
            n.body = n.body[1:] or [ast.Pass()]
    return node

def _parse_function(code):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            tree = ast.parse(code)
        except SyntaxError:                       # e.g. indented method whose dedent failed
            tree = ast.parse("class _Wrapper:\n" + code).body[0]
    fn = next(n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
    return fn

def normalize_function(code):
    fn = _parse_function(code)
    return ast.unparse(_strip_docstrings(fn)) if REMOVE_COMMENTS_DOCSTRINGS else textwrap.dedent(code)

model_inputs, invalid = [], []
for r in df5.itertuples():
    try:
        mi = normalize_function(r.code)
        model_inputs.append(mi if mi.strip() else None)
        if not mi.strip(): invalid.append((r.sample_id, "empty"))
    except Exception as e:
        model_inputs.append(None); invalid.append((r.sample_id, f"{type(e).__name__}"))
df5["code_raw"] = df5.code
df5["model_input"] = model_inputs
invalid_df = pd.DataFrame(invalid, columns=["sample_id", "reason"])
print(f"Invalid samples: {len(invalid_df)}")
if len(invalid_df): display(invalid_df.merge(df5[["sample_id", "label"]], on="sample_id"))
df5 = df5[df5.model_input.notna()].reset_index(drop=True)

# Comment cue on buggy / fixed pairs
n_comment = lambda c: sum(1 for l in c.split("\n") if l.strip().startswith("#"))
pairs = df5[df5.label_source == "buggy_modified"].merge(
    df5[df5.label_source == "fixed_counterpart"], on=["sample_bug_key", "file_path", "qualname"], suffixes=("_b", "_f"))
more_comments = (pairs.code_raw_f.map(n_comment) > pairs.code_raw_b.map(n_comment)).mean()
print(f"Buggy/fixed pairs: {len(pairs)} | fixed version has MORE comment lines in {100 * more_comments:.1f}% of pairs "
      f"(raw code) -> removed by normalization")

ex = pairs[pairs.sample_bug_key == "black-1"].iloc[0] if (pairs.sample_bug_key == "black-1").any() else pairs.iloc[0]
print(f"\nExample — fixed version of {ex.sample_bug_key} :: {ex.qualname}")
print("----- raw (first 14 lines) -----");        print("\n".join(ex.code_raw_f.split("\n")[:14]))
print("----- model input (first 14 lines) -----"); print("\n".join(ex.model_input_f.split("\n")[:14]))

# ---- Verification ----
assert df5.model_input.str.len().gt(0).all()
assert not df5.model_input.str.contains(r"^\s*#", regex=True).any() or not REMOVE_COMMENTS_DOCSTRINGS
print(f"\nSamples after removing invalid: {len(df5)}")
print("[OK] 5.2 passed")

**Interpretation.** If the fixed version has more comment lines in a noticeable share of pairs, raw code would give the classifier a non-semantic cue ("more comments ⇒ fixed ⇒ label 0"). After normalization, the buggy and fixed inputs differ **only in code**, which is exactly the difference that defines the label.

## 5.3 Duplicate removal

**Explanation.** Two samples are duplicates if their **model inputs are identical** (same code after normalization). For each group of duplicates:
* **conflicting labels** (the same code labelled 1 and 0): **all copies are removed**, since the label is ambiguous;
* **same label**: **one** copy is kept (the smallest `sample_id`, deterministic) and the others are removed.

This also removes **cross-split duplicate leakage**: after this step, identical code can no longer appear in both the training and the test set.

**Expected output.** The number of duplicate groups, how many samples were removed for each reason, duplicates across projects and across leakage groups, and `[OK] 5.3 passed`.

In [ ]:
# ---- 5.3 Duplicates ----
g = df5.groupby("model_input")
dup = df5[g.sample_id.transform("size") > 1].copy()
dup["n_labels"] = dup.groupby("model_input").label.transform("nunique")
conflict_ids = set(dup[dup.n_labels > 1].sample_id)
same = dup[dup.n_labels == 1].sort_values("sample_id")
same_drop_ids = set(same[same.duplicated("model_input", keep="first")].sample_id)

dup_groups = dup.groupby("model_input").agg(size=("sample_id", "size"), n_labels=("label", "nunique"),
                                             n_projects=("project", "nunique"), n_groups=("leakage_group", "nunique"),
                                             label_sources=("label_source", lambda s: ", ".join(sorted(set(s)))))
print(f"Duplicate groups: {len(dup_groups)} (samples involved: {len(dup)})")
print(f"  with conflicting labels -> removed entirely : {(dup_groups.n_labels > 1).sum()} groups, {len(conflict_ids)} samples")
print(f"  same label -> extra copies removed          : {len(same_drop_ids)} samples")
print(f"  groups spanning >1 project                  : {(dup_groups.n_projects > 1).sum()}")
print(f"  groups spanning >1 leakage group            : {(dup_groups.n_groups > 1).sum()}  (would leak across splits)")
display(dup_groups.reset_index(drop=True).sort_values("size", ascending=False).head(10))

if conflict_ids:
    print("Conflicting-label samples (removed):")
    display(df5[df5.sample_id.isin(conflict_ids)][["sample_id", "label", "label_source"]])

dedup_log = pd.concat([pd.DataFrame({"sample_id": sorted(conflict_ids), "reason": "duplicate_conflicting_labels"}),
                       pd.DataFrame({"sample_id": sorted(same_drop_ids), "reason": "duplicate_same_label"}),
                       invalid_df.assign(reason="invalid_" + invalid_df.reason)], ignore_index=True)
n_before = len(df5)
df5 = df5[~df5.sample_id.isin(conflict_ids | same_drop_ids)].reset_index(drop=True)
print(f"\nSamples: {n_before} -> {len(df5)}")

# ---- Verification ----
assert df5.model_input.is_unique, "Duplicates remain!"
print("[OK] 5.3 passed")

## 5.4 Code length analysis

**Explanation.** Length of the **model input** in lines and characters, per label source. We recompute the descriptive length-only ROC-AUC (0.5 = length carries no label information) on the normalized inputs.

**Expected output.** A statistics table and two AUC values close to 0.5.

In [ ]:
# ---- 5.4 Code length ----
df5["n_lines_input"] = df5.model_input.str.count("\n") + 1
df5["n_chars_input"] = df5.model_input.str.len()
display(df5.groupby("label_source")[["n_lines_input", "n_chars_input"]]
        .describe(percentiles=[.5, .9]).round(1).T)

def auc_len(col, sources=None):
    d = df5 if sources is None else df5[df5.label_source.isin(sources)]
    return round(roc_auc_score(d.label, d[col]), 3)
len_auc5 = {"lines_all": auc_len("n_lines_input"),
            "lines_buggy_vs_unchanged": auc_len("n_lines_input", ["buggy_modified", "unchanged_same_file"]),
            "chars_all": auc_len("n_chars_input"),
            "chars_buggy_vs_unchanged": auc_len("n_chars_input", ["buggy_modified", "unchanged_same_file"])}
print("Descriptive length-only ROC-AUC:", len_auc5)

# ---- Verification ----
assert (df5.n_lines_input >= 2).all()
print("\n[OK] 5.4 passed")

## 5.5 CodeBERT tokenizer

**Explanation.** We load the tokenizer of `microsoft/codebert-base`, a byte-level BPE vocabulary inherited from RoBERTa. Special tokens: `<s>` (start, id 0), `</s>` (end, id 2), `<pad>` (id 1). The tokenizer is saved to `models/codebert/tokenizer/` on Drive, so the exact same tokenizer is reused in Phase 8.

**Expected output.** `RobertaTokenizerFast`, vocabulary size 50265, max length 512, the tokens of a short example function, and the save path.

In [ ]:
# ---- 5.5 Tokenizer ----
MODEL_NAME = CONFIG["model_name"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
TOK_DIR = PATHS["codebert"] / "tokenizer"
tokenizer.save_pretrained(TOK_DIR)

print("Tokenizer      :", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Max length     :", tokenizer.model_max_length)
print("Special tokens :", {t: tokenizer.convert_tokens_to_ids(t) for t in ["<s>", "</s>", "<pad>", "<unk>"]})
example = "def add(a, b):\n    return a + b"
enc = tokenizer(example)
print(f"\nExample: {example!r}")
print("Tokens   :", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("Saved to :", TOK_DIR)

# ---- Verification ----
assert tokenizer.vocab_size == 50265 and enc["input_ids"][0] == 0 and enc["input_ids"][-1] == 2
print("\n[OK] 5.5 passed")

**Interpretation.** `Ġ` marks a preceding space, and `Ċ` a newline. Indentation spaces also become tokens, which is one reason why Python functions quickly reach 512 tokens.

## 5.6 Token length analysis

**Explanation.** We count the tokens of every model input (including `<s>` and `</s>`, **without** truncation), compare the distributions per label source, and measure the share of samples longer than 512 tokens. The figure is saved to `figures/phase5_token_length.png`.

**Expected output.** A statistics table, the share > 512 per label source, the descriptive token-length AUC, and a histogram with a line at 512.

In [ ]:
# ---- 5.6 Token lengths ----
MAX_LEN = CONFIG["hyperparameters"]["max_seq_length"]       # 512
df5["n_tokens"] = [len(ids) for ids in tokenizer(df5.model_input.tolist(), add_special_tokens=True)["input_ids"]]
df5["truncated"] = df5.n_tokens > MAX_LEN

tok_stats = df5.groupby("label_source").n_tokens.describe(percentiles=[.25, .5, .75, .9]).round(0)
tok_stats["% > 512"] = (100 * df5.groupby("label_source").truncated.mean()).round(1)
display(tok_stats)
print(f"All samples: median {df5.n_tokens.median():.0f} tokens | > {MAX_LEN}: {100 * df5.truncated.mean():.1f}%")
tok_auc = {"all": round(roc_auc_score(df5.label, df5.n_tokens), 3),
           "buggy_vs_unchanged": round(roc_auc_score(*[df5[df5.label_source.isin(["buggy_modified", "unchanged_same_file"])][c]
                                                         for c in ("label", "n_tokens")]), 3)}
print("Descriptive token-length ROC-AUC:", tok_auc)

COLOR_1, COLOR_2 = "#2a78d6", "#eb6834"
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.logspace(np.log10(max(df5.n_tokens.min(), 8)), np.log10(df5.n_tokens.max()), 40)
ax.hist(df5[df5.label == 0].n_tokens, bins=bins, histtype="step", lw=2, color=COLOR_1, label="clean (0)")
ax.hist(df5[df5.label == 1].n_tokens, bins=bins, histtype="step", lw=2, color=COLOR_2, label="buggy (1)")
ax.axvline(MAX_LEN, color="#333333", lw=1, ls="--"); ax.text(MAX_LEN * 1.05, ax.get_ylim()[1] * 0.9, "512 (CodeBERT limit)", fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("Tokens per function (log scale)"); ax.set_ylabel("Number of samples")
ax.set_title("Token length of model inputs", loc="left"); ax.legend(frameon=False); ax.grid(axis="x", visible=False)
fig_path = PATHS["figures"] / "phase5_token_length.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight"); print("Figure saved:", fig_path); plt.show()

# ---- Verification ----
assert (df5.n_tokens >= 3).all()
print("\n[OK] 5.6 passed")

## 5.7 Long sequences: truncation and bug visibility

**Explanation.** Functions longer than 512 tokens are **truncated at the end** (head truncation: the first 510 code tokens are kept). Is the bug still visible to the model? For every **buggy function that has its fixed counterpart** in the dataset, we locate the first line that differs between the two versions and compute its **token position** in the buggy input. If that position is beyond the window, the bug is **invisible** to CodeBERT.

Important: this information is **only measured**, never used to choose the window. Choosing the window around the bug would leak the label, because at prediction time the bug location is unknown.

New columns: `truncated`, `first_change_token`, `bug_beyond_window` (kept for the error analysis in Phase 10).

**Expected output.** The share of truncated samples per class, and the number and share of paired buggy functions whose first change lies beyond the 512-token window.

In [ ]:
# ---- 5.7 Truncation & bug visibility ----
WINDOW = MAX_LEN - 2                       # code tokens kept (without <s> and </s>)
pairs5 = df5[df5.label_source == "buggy_modified"].merge(
    df5[df5.label_source == "fixed_counterpart"][["sample_bug_key", "file_path", "qualname", "model_input"]],
    on=["sample_bug_key", "file_path", "qualname"], suffixes=("", "_fixed"))

def first_change_token(buggy, fixed):
    b, f = buggy.split("\n"), fixed.split("\n")
    sm = difflib.SequenceMatcher(None, b, f, autojunk=False)
    first = next((i1 for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"), None)
    if first is None: return np.nan
    prefix = "\n".join(b[:first])
    return len(tokenizer(prefix, add_special_tokens=False)["input_ids"]) + 1   # +1 for <s>

pairs5["first_change_token"] = [first_change_token(b, f) for b, f in zip(pairs5.model_input, pairs5.model_input_fixed)]
df5 = df5.drop(columns=["first_change_token"], errors="ignore").merge(
    pairs5[["sample_id", "first_change_token"]], on="sample_id", how="left")
df5["bug_beyond_window"] = df5.first_change_token > WINDOW

print("Truncated samples (> 512 tokens):")
display((100 * df5.groupby("label_source").truncated.mean()).round(1).to_frame("% truncated"))
paired = df5[df5.first_change_token.notna()]
print(f"Buggy functions with a fixed counterpart: {len(paired)}")
print(f"  first change beyond the 512-token window: {paired.bug_beyond_window.sum()} "
      f"({100 * paired.bug_beyond_window.mean():.1f}%)  -> bug invisible to CodeBERT")
print(f"  median token position of the first change: {paired.first_change_token.median():.0f}")

# ---- Verification ----
assert paired.first_change_token.ge(1).all()
print("\n[OK] 5.7 passed")

**Interpretation.** The share of paired buggy functions with an *invisible* bug is a **ceiling on what any 512-token model can learn** for those samples. It is reported as a limitation. If the share is large, Phase 8 can compare head truncation with a **chunking** strategy (split long functions into 512-token windows and aggregate the predictions), which does not need the bug location.

## 5.8 Tokenization (padding, truncation) and saving

**Explanation.** All model inputs are tokenized with `padding="max_length"`, `truncation=True` and `max_length=512`, producing `input_ids` and `attention_mask` arrays of shape `(n_samples, 512)`.

Saved files (`data/processed/`):
* `phase5_dataset_v2.jsonl` / `.csv`: one row per sample, with all identifiers, metadata, flags and `model_input`;
* `phase5_tokenized_v2.npz`: `sample_id`, `input_ids`, `attention_mask` and `labels`, aligned row by row;
* `phase5_removed_samples.csv`: every removed sample and the reason.

**Expected output.** Array shapes, padding and special-token checks, the saved paths, and `[OK] 5.8 passed`.

In [ ]:
# ---- 5.8 Tokenize & save ----
df5 = df5.sort_values("sample_id").reset_index(drop=True)
enc = tokenizer(df5.model_input.tolist(), padding="max_length", truncation=True, max_length=MAX_LEN,
                return_tensors="np")
input_ids = enc["input_ids"].astype(np.int32); attention_mask = enc["attention_mask"].astype(np.int8)
labels = df5.label.to_numpy(np.int8)
print("input_ids     :", input_ids.shape, input_ids.dtype)
print("attention_mask:", attention_mask.shape, attention_mask.dtype)
print(f"Mean real tokens per sample: {attention_mask.sum(1).mean():.0f} / {MAX_LEN}")

out5 = {"jsonl": PATHS["processed"] / "phase5_dataset_v2.jsonl",
        "csv": PATHS["processed"] / "phase5_dataset_v2.csv",
        "npz": PATHS["processed"] / "phase5_tokenized_v2.npz",
        "removed": PATHS["processed"] / "phase5_removed_samples.csv"}
df5.to_json(out5["jsonl"], orient="records", lines=True, force_ascii=False)
df5.to_csv(out5["csv"], index=False)
np.savez_compressed(out5["npz"], sample_id=df5.sample_id.to_numpy(), input_ids=input_ids,
                    attention_mask=attention_mask, labels=labels)
dedup_log.to_csv(out5["removed"], index=False)
for k, p in out5.items(): print(f"{k:<8} -> {p}")

# ---- Verification ----
z = np.load(out5["npz"], allow_pickle=True)
assert z["input_ids"].shape == (len(df5), MAX_LEN) and (z["sample_id"] == df5.sample_id.to_numpy()).all()
assert (input_ids[:, 0] == tokenizer.cls_token_id).all()
real_len = attention_mask.sum(1)
assert (input_ids[np.arange(len(df5)), real_len - 1] == tokenizer.sep_token_id).all(), "</s> missing"
assert ((input_ids == tokenizer.pad_token_id) == (attention_mask == 0)).all(), "padding/mask mismatch"
assert (real_len == np.minimum(df5.n_tokens, MAX_LEN)).all()
print("\n[OK] 5.8 passed")

## 5.9 Phase 5 summary

**Expected output.** The preprocessing funnel, the final class distribution, the configuration update, and a summary table with `PASS` on every row.

In [ ]:
# ---- 5.9 Summary ----
funnel5 = pd.DataFrame({"step": ["Phase 4 dataset (v2)", "after removing invalid samples",
                                 "after removing duplicates (final)"],
                        "n_samples": [len(pd.read_json(data4_path, lines=True)), n_before, len(df5)]})
display(funnel5)
final_dist = df5.groupby(["label", "label_source"]).size().rename("n").reset_index()
display(final_dist)
print(f"Final: {len(df5)} samples | buggy (1): {(df5.label == 1).sum()} ({100 * df5.label.mean():.1f}%) | "
      f"clean (0): {(df5.label == 0).sum()} | projects: {df5.project.nunique()} | bugs: {df5.sample_bug_key.nunique()} "
      f"| leakage groups: {df5.leakage_group.nunique()}")

CONFIG["preprocessing"] = {
    "input": "function without comments/docstrings, canonical formatting (ast.unparse)" if REMOVE_COMMENTS_DOCSTRINGS else "raw dedented function",
    "remove_comments_docstrings": REMOVE_COMMENTS_DOCSTRINGS,
    "dedup": "identical model_input; conflicting labels -> all removed; same label -> keep smallest sample_id",
    "tokenizer": MODEL_NAME, "tokenizer_dir": str(TOK_DIR),
    "max_length": MAX_LEN, "truncation": "head (keep first 510 code tokens)", "padding": "max_length",
    "n_invalid": len(invalid_df), "n_duplicate_conflicting": len(conflict_ids), "n_duplicate_same_label": len(same_drop_ids),
    "n_final": len(df5), "n_positive": int(df5.label.sum()), "n_negative": int((df5.label == 0).sum()),
    "pct_truncated": round(100 * df5.truncated.mean(), 1),
    "pct_paired_bug_beyond_window": round(100 * paired.bug_beyond_window.mean(), 1),
    "token_length_auc": tok_auc, "line_length_auc": len_auc5,
    "files": {k: str(v) for k, v in out5.items()},
}
save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
save_json({"run_id": RUN_ID, **CONFIG["preprocessing"], "funnel": funnel5.to_dict("records")},
          PATHS["metrics"] / "phase5_summary.json")

checks5 = {
    "Dataset v2 loaded":                         "PASS" if "v2_length_matched" in df5.dataset_version.unique() else "FAIL",
    "Invalid samples removed":                   "PASS",
    "No duplicate model inputs":                 "PASS" if df5.model_input.is_unique else "FAIL",
    "No conflicting labels":                     "PASS" if df5.groupby("model_input").label.nunique().max() == 1 else "FAIL",
    "Tokenizer saved":                           "PASS" if (TOK_DIR / "tokenizer_config.json").exists() else "FAIL",
    "Tokenized arrays (n x 512) saved":          "PASS" if out5["npz"].exists() else "FAIL",
    "Identifiers & metadata preserved":          "PASS" if {"project", "bug_id", "file_path", "commit_sha", "leakage_group"} <= set(df5.columns) else "FAIL",
    "Length shortcut small (token AUC < 0.6)":   "PASS" if max(tok_auc.values()) < 0.6 else "WARN",
}
print()
print(pd.DataFrame({"check": list(checks5), "status": list(checks5.values())}).to_string(index=False))
assert "FAIL" not in checks5.values()
print(f"\n[OK] PHASE 5 COMPLETE — {len(df5)} samples ready for splitting "
      f"({int(df5.label.sum())} buggy / {int((df5.label == 0).sum())} clean).")

**Please report back** the outputs of **5.2** (invalid samples, comment cue), **5.3** (duplicates), **5.6** (token statistics and figure), **5.7** (bug beyond window) and **5.9** (summary).

➡️ Continue with **PHASE 6** below.

---
# PHASE 6 — Leakage-free Dataset Splitting

> **After a Colab restart:** Phase 6 only needs **Phase 1** (cells 1.1 → 1.8, with `RESUME_RUN_ID` set in 1.7). It reads the Phase 5 outputs from Drive.

### Decisions taken after Phase 5 (delegated to the assistant by the researcher, with justification)
| Question | Decision | Justification |
|---|---|---|
| Residual length difference after normalization (token-length AUC 0.525 overall, 0.559 buggy vs unchanged) | **keep dataset v2** | small effect, measured directly by the **length-only baseline** in Phase 7 |
| Functions longer than 512 tokens (bug beyond the window in 8.3% of paired buggy functions) | **head truncation only** | standard for CodeBERT. The affected samples are flagged (`bug_beyond_window`) and analysed separately in Phase 10 |
| Split protocol | **5-fold cross-project cross-validation** (instead of a single 70/15/15 split) | see below |

### Why not a single 70/15/15 project-level split?
After Phase 5, **pandas holds 31% of the samples**, and six projects have fewer than 25 samples each. A 15% test set (≈ 270 samples) would contain only 2–4 projects. The measured performance would then depend mainly on *which* projects were drawn, which makes the result unstable and hard to defend.

### Protocol: grouped 5-fold cross-validation by project
* **Outer folds (test):** the 17 projects are partitioned into 5 folds. **pandas forms its own fold**. The 16 other projects are assigned to 4 folds balanced by sample count (greedy: largest project first, into the currently smallest fold). **Each project is tested exactly once**, by a model that never saw it.
* **Inner validation (early stopping, model selection):** inside each training part, a set of **whole projects** (≈ 15% of the training samples, pandas excluded) is held out as validation, so validation is also cross-project. The choice is deterministic (closest total to 15%, ties broken by class balance).
* **Leakage prevention, verified in 6.4:** no project, leakage group, bug, sample or identical code is shared between train, validation and test of a fold.
* The **same partitions** are used by every model (baselines in Phase 7, CodeBERT in Phase 8).
* Results are reported as **mean ± standard deviation across the 5 folds**, plus metrics on the pooled out-of-fold predictions (every sample predicted exactly once).

Sections: 6.1 Load · 6.2 Outer folds · 6.3 Validation projects · 6.4 Leakage verification · 6.5 Save split identifiers & summary

## 6.1 Load the Phase 5 dataset

**Expected output.** 1799 samples, 17 projects, the tokenized arrays aligned with the table, and `[OK] 6.1 passed`.

In [ ]:
# ---- 6.1 Load Phase 5 outputs ----
_missing = [n for n in ("PATHS", "CONFIG", "RUN_ID", "SEED", "save_json") if n not in globals()]
if _missing:
    raise RuntimeError(f"Run Phase 1 first (cells 1.1 -> 1.8, with RESUME_RUN_ID in 1.7). Missing: {_missing}")
import itertools
cfg_file = PATHS["logs"] / f"config_{RUN_ID}.json"
saved_cfg = json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
for k in ("dataset", "labeling", "planned_baselines", "preprocessing"):
    if k in saved_cfg: CONFIG[k] = saved_cfg[k]

df6 = pd.read_json(PATHS["processed"] / "phase5_dataset_v2.jsonl", lines=True, dtype={"commit_sha": str})
z6 = np.load(PATHS["processed"] / "phase5_tokenized_v2.npz", allow_pickle=True)
print(f"Samples: {len(df6)} | projects: {df6.project.nunique()} | bugs: {df6.sample_bug_key.nunique()} | "
      f"leakage groups: {df6.leakage_group.nunique()} | buggy: {100 * df6.label.mean():.1f}%")

# ---- Verification ----
assert (z6["sample_id"] == df6.sample_id.to_numpy()).all(), "Tokenized arrays not aligned with the table"
assert df6.sample_id.is_unique and df6.model_input.is_unique
print("\n[OK] 6.1 passed")

## 6.2 Outer folds: projects → 5 test folds

**Explanation.** pandas is placed alone in one fold. The other projects are sorted by size and assigned one by one to the fold that currently has the fewest samples. Ties are broken by project name, so the assignment is deterministic.

**Expected output.** The project-to-fold table and a per-fold summary: samples, share, buggy % and number of projects. Folds 1–4 should have ≈ 310 samples each.

In [ ]:
# ---- 6.2 Outer folds ----
N_FOLDS, ANCHOR = 5, "pandas"
proj_sizes = df6.groupby("project").size().sort_values(ascending=False)
fold_of = {ANCHOR: 0}
loads = {k: 0 for k in range(1, N_FOLDS)}
for p in sorted(proj_sizes.index.drop(ANCHOR), key=lambda p: (-proj_sizes[p], p)):
    k = min(loads, key=lambda k: (loads[k], k))
    fold_of[p] = k; loads[k] += int(proj_sizes[p])
df6["fold"] = df6.project.map(fold_of)

fold_tbl = df6.groupby("fold").agg(samples=("sample_id", "size"), buggy_pct=("label", lambda s: round(100 * s.mean(), 1)),
                                   n_projects=("project", "nunique"), n_bugs=("sample_bug_key", "nunique"),
                                   projects=("project", lambda s: ", ".join(sorted(set(s)))))
fold_tbl.insert(1, "share_%", (100 * fold_tbl.samples / len(df6)).round(1))
display(fold_tbl)

# ---- Verification ----
assert df6.groupby("project").fold.nunique().max() == 1, "a project is split across folds"
assert set(df6.fold) == set(range(N_FOLDS)) and df6.groupby("fold").label.nunique().min() == 2
print("[OK] 6.2 passed")

**Interpretation.** Fold 0 (pandas) is the largest test fold, so its result tells us how well a model trained on the 16 other projects transfers to the biggest and most complex project. The class ratio should be similar across folds (≈ 40–50% buggy). Otherwise, F1 and ROC-AUC differences between folds could partly reflect class balance.

## 6.3 Validation projects inside each training part

**Explanation.** For each outer fold *k*, the training part is made of the projects of the 4 other folds. Among them (pandas excluded), we choose the **set of whole projects** whose total sample count is closest to **15%** of the training part. Every subset is checked exhaustively, and ties are broken by the buggy share closest to the training part's. These projects become the **validation** set (early stopping and model selection); the rest is **train**.

**Expected output.** For each fold: train / validation / test sizes, buggy %, and the validation projects.

In [ ]:
# ---- 6.3 Inner validation projects ----
VAL_FRACTION = 0.15
split_rows, inner = [], {}
for k in range(N_FOLDS):
    trv = df6[df6.fold != k]
    cand = sorted(p for p in trv.project.unique() if p != ANCHOR)
    sizes = trv.groupby("project").size(); pos = trv.groupby("project").label.sum()
    target, base_rate = VAL_FRACTION * len(trv), trv.label.mean()
    best = None
    for r in range(1, len(cand) + 1):
        for combo in itertools.combinations(cand, r):
            n = sizes[list(combo)].sum()
            if n > 2 * target: continue
            rate = pos[list(combo)].sum() / n
            score = (abs(n - target), abs(rate - base_rate), combo)
            if best is None or score < best: best = score
    val_projects = set(best[2]); inner[k] = sorted(val_projects)
    split = np.where(df6.fold == k, "test", np.where(df6.project.isin(val_projects), "val", "train"))
    split_rows.append(pd.DataFrame({"sample_id": df6.sample_id, "fold": k, "split": split}))

splits = pd.concat(split_rows, ignore_index=True)
summ = (splits.merge(df6[["sample_id", "label"]], on="sample_id")
              .groupby(["fold", "split"]).agg(n=("sample_id", "size"), buggy_pct=("label", lambda s: round(100 * s.mean(), 1)))
              .unstack("split"))
summ.columns = [f"{a}_{b}" for a, b in summ.columns]
summ = summ[[f"n_{s}" for s in ("train", "val", "test")] + [f"buggy_pct_{s}" for s in ("train", "val", "test")]]
summ["val_projects"] = [", ".join(inner[k]) for k in summ.index]
summ["test_projects"] = [", ".join(sorted(df6[df6.fold == k].project.unique())) for k in summ.index]
display(summ)
pct = splits.groupby(["fold", "split"]).size().unstack().div(len(df6)).mul(100).round(1)
print("Share of all samples per split (%):"); display(pct[["train", "val", "test"]])

# ---- Verification ----
assert (splits.groupby("fold").size() == len(df6)).all()
assert splits.groupby(["fold", "split"]).size().unstack().gt(0).all().all()
print("[OK] 6.3 passed")

## 6.4 Leakage verification

**Explanation.** For every fold we verify that **train, validation and test share nothing**:
* projects, leakage groups, bugs (`sample_bug_key`), samples (`sample_id`), and identical model inputs;
* across the 5 folds, every sample is in **test exactly once**.

**Expected output.** A table with zero overlaps for every fold and check, and `[OK] 6.4 passed`.

In [ ]:
# ---- 6.4 Leakage verification ----
chk_rows = []
meta = df6.set_index("sample_id")
for k in range(N_FOLDS):
    s = splits[splits.fold == k].set_index("sample_id").split
    part = {name: meta.loc[s[s == name].index] for name in ("train", "val", "test")}
    row = {"fold": k}
    for col in ("project", "leakage_group", "sample_bug_key", "model_input"):
        sets = {n: set(p[col]) for n, p in part.items()}
        row[f"{col}_overlap"] = len(sets["train"] & sets["test"]) + len(sets["train"] & sets["val"]) + len(sets["val"] & sets["test"])
    chk_rows.append(row)
leak_tbl = pd.DataFrame(chk_rows).set_index("fold")
display(leak_tbl)
test_count = splits[splits.split == "test"].sample_id.value_counts()
print("Every sample tested exactly once:", bool((test_count == 1).all() and len(test_count) == len(df6)))

# ---- Verification ----
assert (leak_tbl == 0).all().all(), "LEAKAGE detected!"
assert (test_count == 1).all() and len(test_count) == len(df6)
print("\n[OK] 6.4 passed — no project, group, bug or duplicate leakage")

## 6.5 Save split identifiers and Phase 6 summary

**Explanation.** The split is saved in a **long format** (`sample_id`, `fold`, `split`). Its SHA-256 fingerprint is recorded in the configuration, so later phases can verify that they use exactly the same partitions.

Files:
* `data/processed/phase6_splits.csv`: one row per (sample, fold);
* `data/processed/phase6_folds.json`: the projects of each fold (test / validation);
* `results/metrics/phase6_split_summary.csv`.

**Expected output.** The saved paths, the split fingerprint, and a summary table with `PASS` on every row.

In [ ]:
# ---- 6.5 Save & summary ----
out6 = {"splits": PATHS["processed"] / "phase6_splits.csv",
        "folds": PATHS["processed"] / "phase6_folds.json",
        "summary": PATHS["metrics"] / "phase6_split_summary.csv"}
splits.sort_values(["fold", "sample_id"]).to_csv(out6["splits"], index=False)
split_sha = sha256_file(out6["splits"]) if "sha256_file" in globals() else hashlib.sha256(out6["splits"].read_bytes()).hexdigest()
folds_json = {"protocol": "grouped 5-fold cross-project CV", "anchor_project_own_fold": ANCHOR,
              "val_fraction_of_training_part": VAL_FRACTION,
              "folds": {int(k): {"test_projects": sorted(df6[df6.fold == k].project.unique()),
                                 "val_projects": inner[k]} for k in range(N_FOLDS)},
              "splits_sha256": split_sha}
save_json(folds_json, out6["folds"])
summ.to_csv(out6["summary"])

CONFIG["decisions_after_phase5"] = {
    "length_residual": "keep dataset v2 (verified by length-only baseline in Phase 7)",
    "long_functions": "head truncation only; bug_beyond_window samples analysed in Phase 10",
    "split": "grouped 5-fold cross-project CV (assistant's recommendation, delegated by the researcher)"}
CONFIG["split"] = {"strategy": "grouped 5-fold cross-project cross-validation; pandas = own fold; "
                               "inner validation = whole projects (~15% of training part)",
                   "n_folds": N_FOLDS, "val_fraction": VAL_FRACTION,
                   "split_ids_file": str(out6["splits"]), "folds_file": str(out6["folds"]), "splits_sha256": split_sha,
                   "replaces": "single 70/15/15 project split (unreliable: pandas = 31% of samples)"}
save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
for k, p in out6.items(): print(f"{k:<8} -> {p}")
print("Split fingerprint (SHA-256):", split_sha)

checks6 = {
    "5 folds, each project in exactly one test fold": "PASS" if df6.groupby("project").fold.nunique().max() == 1 else "FAIL",
    "Every sample tested exactly once":               "PASS" if (test_count == 1).all() else "FAIL",
    "Validation = whole held-out projects":           "PASS",
    "No project / group / bug / duplicate leakage":   "PASS" if (leak_tbl == 0).all().all() else "FAIL",
    "Both classes in every train/val/test part":      "PASS" if splits.merge(df6[["sample_id", "label"]]).groupby(["fold", "split"]).label.nunique().min() == 2 else "FAIL",
    "Split identifiers saved + fingerprint":          "PASS" if out6["splits"].exists() else "FAIL",
}
print()
print(pd.DataFrame({"check": list(checks6), "status": list(checks6.values())}).to_string(index=False))
assert "FAIL" not in checks6.values()
print("\n[OK] PHASE 6 COMPLETE — 5 cross-project folds ready (same partitions for baselines and CodeBERT).")

**Please report back** the tables of **6.2** (folds), **6.3** (train / val / test per fold), **6.4** (leakage check) and the summary of **6.5**.

➡️ **PHASE 7 (Baselines: TF-IDF + LR, TF-IDF + RF, length-only)** will be added after the Phase 6 results are confirmed.